# AI Model 03 데이터 격자화


In [ ]:
# ============================================================
# AI Model 03 공통 경로 설정
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
DATA_PATH = os.path.join(ROOT_PATH, "data") + "/"
PROCESSED_PATH = os.path.join(ROOT_PATH, "preprocessed_data") + "/"
DATA_COLLECTION_PATH = os.path.join(ROOT_PATH, "data collection") + "/"

print(f"ROOT_PATH      = {ROOT_PATH}")
print(f"DATA_PATH      = {DATA_PATH}")
print(f"PROCESSED_PATH = {PROCESSED_PATH}")
print(f"DATA_COLLECTION_PATH = {DATA_COLLECTION_PATH}")

!pip -q install geopandas pyogrio shapely pyproj duckdb pyarrow scipy matplotlib


### 그리드 작업 (그리드 생성 및 데이터 입력)

#### 전체 육지 500m grid 만들기

In [ ]:
# ============================================================
# 1셀. 전체 육지 500m grid 만들기
# - Natural Earth minor islands (작은 파일) + 제주/울릉/독도 guard
# - 주유소 metadata location history patch
# - 시설 위치 patch
# - placeholder 컬럼은 만들지 않음
# 결과:
#   1) south_korea_land_mask_500m.gpkg
#   2) korea_land_grid_500m.parquet
# ============================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import requests

from shapely.geometry import Point
from shapely.ops import unary_union
from pyproj import Transformer

warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------
# 기본 설정
# ------------------------------------------------------------
CELL_SIZE_M = 500
SOURCE_CRS = "EPSG:4326"
WORK_CRS   = "EPSG:5179"

TO_WORK = Transformer.from_crs(SOURCE_CRS, WORK_CRS, always_xy=True)
TO_WGS  = Transformer.from_crs(WORK_CRS, SOURCE_CRS, always_xy=True)

DATA1_DIR = Path(ROOT_PATH) / "그리드" / "data_1"
DATA1_DIR.mkdir(parents=True, exist_ok=True)

REGION_ROOT = Path(PROCESSED_PATH) / "additional_data" / "gas_station_prices_by_region"
FACILITY_CSV_PATH = Path(PROCESSED_PATH) / "additional_data" / "1 facility_location_data_final.csv"

LAND_MASK_PATH = DATA1_DIR / f"south_korea_land_mask_{CELL_SIZE_M}m.gpkg"
LAND_GRID_PATH = DATA1_DIR / f"korea_land_grid_{CELL_SIZE_M}m.parquet"

NE_ZIP_PATH = Path("/content/ne_10m_admin_0_scale_rank_minor_islands.zip")
NE_URLS = [
    "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_scale_rank_minor_islands.zip",
    "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_scale_rank_minor_islands.zip",
]

# 작은 섬 보정
GUARD_ISLANDS = [
    {"name": "Jeju",    "lon": 126.5312, "lat": 33.4996, "radius_m": 35000},
    {"name": "Ulleung", "lon": 130.9057, "lat": 37.4846, "radius_m": 7000},
    {"name": "Dokdo",   "lon": 131.8702, "lat": 37.2418, "radius_m": 1000},
]

# 이후 셀에서 재사용할 분류 기준
BRAND_ORDER = [
    "SK에너지",
    "GS칼텍스",
    "HD현대오일뱅크",
    "S-OIL",
    "알뜰",
    "NH-OIL",
    "자가상표",
    "기타",
]
SELF_ORDER = ["셀프", "일반", "미상"]

# ------------------------------------------------------------
# 공통 함수
# ------------------------------------------------------------
def ql(s: str) -> str:
    return "'" + str(s).replace("'", "''") + "'"

def qi(s: str) -> str:
    return '"' + str(s).replace('"', '""') + '"'

def read_csv_flexible(path, **kwargs):
    path = Path(path)
    last_error = None
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            last_error = e
    raise last_error

def normalize_brand(x):
    if pd.isna(x):
        return "기타"
    s = str(x).strip()
    if not s:
        return "기타"
    s = s.replace("현대오일뱅크", "HD현대오일뱅크")
    if "HD현대오일뱅크" in s:
        return "HD현대오일뱅크"
    if "GS칼텍스" in s:
        return "GS칼텍스"
    if "SK에너지" in s or s == "SK":
        return "SK에너지"
    if "S-OIL" in s or "에쓰오일" in s:
        return "S-OIL"
    if "NH-OIL" in s:
        return "NH-OIL"
    if "알뜰" in s:
        return "알뜰"
    if "자가상표" in s:
        return "자가상표"
    return "기타"

def normalize_self(x):
    if pd.isna(x):
        return "미상"
    s = str(x).strip()
    if "셀프" in s:
        return "셀프"
    if s:
        return "일반"
    return "미상"

def normalize_facility_type(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if "저유소" in s:
        return "storage"
    if "공장" in s:
        return "factory"
    if "대리점" in s:
        return "agency"
    return np.nan

def snap_down(v, cell):
    return int(np.floor(v / cell) * cell)

def snap_up(v, cell):
    return int(np.ceil(v / cell) * cell)

def add_center_and_grid_id(df, cell_size_m=500):
    out = df.copy()
    out["cell_x"] = out["cell_x"].astype("int32")
    out["cell_y"] = out["cell_y"].astype("int32")

    out["center_x"] = out["cell_x"].astype("float64") + cell_size_m / 2.0
    out["center_y"] = out["cell_y"].astype("float64") + cell_size_m / 2.0

    center_lon, center_lat = TO_WGS.transform(
        out["center_x"].to_numpy(),
        out["center_y"].to_numpy(),
    )
    out["center_lon"] = np.round(center_lon, 8)
    out["center_lat"] = np.round(center_lat, 8)

    out["grid_col"] = (out["cell_x"] // cell_size_m).astype("int64")
    out["grid_row"] = (out["cell_y"] // cell_size_m).astype("int64")
    out["grid_id"] = [
        f"G{cell_size_m}_{c}_{r}"
        for c, r in zip(out["grid_col"], out["grid_row"])
    ]

    return out[[
        "grid_id",
        "cell_x", "cell_y",
        "center_x", "center_y",
        "center_lon", "center_lat",
        "grid_col", "grid_row",
    ]]

def distinct_grid_from_lonlat(df, lon_col="lon", lat_col="lat", cell_size_m=500):
    work = df.copy()
    work[lon_col] = pd.to_numeric(work[lon_col], errors="coerce")
    work[lat_col] = pd.to_numeric(work[lat_col], errors="coerce")

    work = work.dropna(subset=[lon_col, lat_col]).copy()
    work = work[
        work[lon_col].between(120, 135) &
        work[lat_col].between(30, 45)
    ].copy()

    if len(work) == 0:
        return pd.DataFrame(columns=[
            "grid_id", "cell_x", "cell_y",
            "center_x", "center_y",
            "center_lon", "center_lat",
            "grid_col", "grid_row",
        ])

    x, y = TO_WORK.transform(work[lon_col].to_numpy(), work[lat_col].to_numpy())
    cell_x = (np.floor(np.asarray(x) / cell_size_m).astype(np.int64) * cell_size_m).astype("int32")
    cell_y = (np.floor(np.asarray(y) / cell_size_m).astype(np.int64) * cell_size_m).astype("int32")

    out = pd.DataFrame({"cell_x": cell_x, "cell_y": cell_y}).drop_duplicates().reset_index(drop=True)
    out = add_center_and_grid_id(out, cell_size_m=cell_size_m)
    return out

def download_ne_zip(zip_path, urls):
    zip_path = Path(zip_path)
    if zip_path.exists() and zip_path.stat().st_size > 0:
        print(f"[다운로드 생략] {zip_path}")
        return zip_path

    for url in urls:
        try:
            print(f"[다운로드 시도] {url}")
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(zip_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            if zip_path.exists() and zip_path.stat().st_size > 0:
                print(f"[다운로드 완료] {zip_path} ({zip_path.stat().st_size:,} bytes)")
                return zip_path
        except Exception as e:
            print(f"[다운로드 실패] {e}")

    raise RuntimeError("Natural Earth zip 다운로드 실패")

def load_korea_land_geom_from_ne(zip_path):
    gdf = gpd.read_file(f"zip://{zip_path}")
    if gdf.crs is None:
        gdf = gdf.set_crs(SOURCE_CRS)

    obj_cols = [c for c in gdf.columns if str(gdf[c].dtype) == "object"]
    mask = np.zeros(len(gdf), dtype=bool)

    for c in obj_cols:
        s = gdf[c].astype(str)
        mask |= s.eq("KOR").to_numpy()
        mask |= s.str.contains("South Korea", case=False, na=False).to_numpy()
        mask |= s.str.contains("Republic of Korea", case=False, na=False).to_numpy()
        mask |= s.str.contains("Korea, South", case=False, na=False).to_numpy()

    kor = gdf.loc[mask].copy()
    if len(kor) == 0:
        raise RuntimeError("Natural Earth에서 South Korea feature를 찾지 못했습니다.")

    kor = kor.to_crs(WORK_CRS)
    kor = kor[kor.geometry.notna() & (~kor.geometry.is_empty)].copy()

    land_geom = unary_union(list(kor.geometry))

    guard_patches = []
    for spec in GUARD_ISLANDS:
        x, y = TO_WORK.transform(spec["lon"], spec["lat"])
        p = Point(x, y)
        if not p.within(land_geom):
            guard_patches.append(p.buffer(spec["radius_m"]))
            print(f"[guard island 추가] {spec['name']}")

    if guard_patches:
        land_geom = unary_union([land_geom] + guard_patches)

    land_gdf = gpd.GeoDataFrame(
        {"name": ["south_korea_land"]},
        geometry=[land_geom],
        crs=WORK_CRS,
    )
    return land_gdf

def build_land_grid_from_geom(land_gdf, cell_size_m=500):
    minx, miny, maxx, maxy = land_gdf.total_bounds
    minx = snap_down(minx, cell_size_m)
    miny = snap_down(miny, cell_size_m)
    maxx = snap_up(maxx, cell_size_m)
    maxy = snap_up(maxy, cell_size_m)

    x_vals = np.arange(minx, maxx, cell_size_m, dtype=np.int64)
    y_vals = np.arange(miny, maxy, cell_size_m, dtype=np.int64)

    xx = np.repeat(x_vals, len(y_vals))
    yy = np.tile(y_vals, len(x_vals))

    center_x = xx + cell_size_m / 2.0
    center_y = yy + cell_size_m / 2.0

    cand = gpd.GeoDataFrame(
        {
            "cell_x": xx.astype("int32"),
            "cell_y": yy.astype("int32"),
            "center_x": center_x.astype("float64"),
            "center_y": center_y.astype("float64"),
        },
        geometry=gpd.points_from_xy(center_x, center_y),
        crs=WORK_CRS,
    )

    inside = gpd.sjoin(
        cand,
        land_gdf[["geometry"]],
        how="inner",
        predicate="within",
    ).drop(columns=["index_right", "geometry"])

    inside = pd.DataFrame(inside).reset_index(drop=True)
    inside = add_center_and_grid_id(inside[["cell_x", "cell_y"]], cell_size_m=cell_size_m)
    return inside

def extract_station_patch_grids(region_root, cell_size_m=500):
    region_root = Path(region_root)
    meta_files = sorted(region_root.glob("*/metadata__latlon.json"))

    rows = []
    for fp in meta_files:
        with open(fp, "r", encoding="utf-8") as f:
            meta = json.load(f)

        for info in meta.values():
            for item in info.get("location", []):
                if not item or len(item) < 3:
                    continue
                try:
                    lat = float(item[1])
                    lon = float(item[2])
                except Exception:
                    continue
                rows.append((lon, lat))

    if not rows:
        return pd.DataFrame(columns=[
            "grid_id", "cell_x", "cell_y",
            "center_x", "center_y",
            "center_lon", "center_lat",
            "grid_col", "grid_row",
        ])

    loc_df = pd.DataFrame(rows, columns=["lon", "lat"])
    return distinct_grid_from_lonlat(loc_df, lon_col="lon", lat_col="lat", cell_size_m=cell_size_m)

def extract_facility_patch_grids(facility_csv_path, cell_size_m=500):
    if not Path(facility_csv_path).exists():
        return pd.DataFrame(columns=[
            "grid_id", "cell_x", "cell_y",
            "center_x", "center_y",
            "center_lon", "center_lat",
            "grid_col", "grid_row",
        ])

    fac = read_csv_flexible(facility_csv_path)
    if not set(["경도", "위도"]).issubset(fac.columns):
        return pd.DataFrame(columns=[
            "grid_id", "cell_x", "cell_y",
            "center_x", "center_y",
            "center_lon", "center_lat",
            "grid_col", "grid_row",
        ])

    fac = fac.rename(columns={"경도": "lon", "위도": "lat"}).copy()
    return distinct_grid_from_lonlat(fac, lon_col="lon", lat_col="lat", cell_size_m=cell_size_m)

# ------------------------------------------------------------
# 실행
# ------------------------------------------------------------
print("=" * 100)
print("[1] Natural Earth 작은 파일 다운로드")
download_ne_zip(NE_ZIP_PATH, NE_URLS)

print("=" * 100)
print("[2] 남한 육지 mask 생성")
land_gdf = load_korea_land_geom_from_ne(NE_ZIP_PATH)
land_gdf.to_file(LAND_MASK_PATH, driver="GPKG")
print(f"[저장] {LAND_MASK_PATH}")

print("=" * 100)
print("[3] Natural Earth 기준 육지 grid 생성")
base_land_grid = build_land_grid_from_geom(land_gdf, cell_size_m=CELL_SIZE_M)
print(f" - base land grid 수: {len(base_land_grid):,}")

print("=" * 100)
print("[4] 주유소 metadata location patch")
station_patch_grid = extract_station_patch_grids(REGION_ROOT, cell_size_m=CELL_SIZE_M)
print(f" - station patch grid 수: {len(station_patch_grid):,}")

print("=" * 100)
print("[5] 시설 위치 patch")
facility_patch_grid = extract_facility_patch_grids(FACILITY_CSV_PATH, cell_size_m=CELL_SIZE_M)
print(f" - facility patch grid 수: {len(facility_patch_grid):,}")

print("=" * 100)
print("[6] 최종 land grid 합치기")
final_land_grid = pd.concat(
    [
        base_land_grid[["cell_x", "cell_y"]],
        station_patch_grid[["cell_x", "cell_y"]],
        facility_patch_grid[["cell_x", "cell_y"]],
    ],
    ignore_index=True,
).drop_duplicates().reset_index(drop=True)

final_land_grid = add_center_and_grid_id(final_land_grid, cell_size_m=CELL_SIZE_M)
final_land_grid.to_parquet(LAND_GRID_PATH, index=False)
print(f"[저장] {LAND_GRID_PATH}")

base_keys = base_land_grid[["cell_x", "cell_y"]].drop_duplicates()
station_only = station_patch_grid[["cell_x", "cell_y"]].merge(
    base_keys, on=["cell_x", "cell_y"], how="left", indicator=True
)
station_only = station_only[station_only["_merge"] == "left_only"]

facility_only = facility_patch_grid[["cell_x", "cell_y"]].merge(
    base_keys, on=["cell_x", "cell_y"], how="left", indicator=True
)
facility_only = facility_only[facility_only["_merge"] == "left_only"]

summary_df = pd.DataFrame([{
    "base_land_grid_count": len(base_land_grid),
    "station_patch_grid_count": len(station_patch_grid),
    "facility_patch_grid_count": len(facility_patch_grid),
    "station_patch_only_count": len(station_only),
    "facility_patch_only_count": len(facility_only),
    "final_land_grid_count": len(final_land_grid),
}])

display(summary_df)
display(final_land_grid.head(10))

#### 주유소 기본 패널 생성

In [ ]:
# ============================================================
# A 셀. 주유소 기본 패널 생성
# - 주유소 영향력(station_neighbor_influence) 제외
# - 결과:
#   grid_station_daily_panel_500m.parquet
# ============================================================

import os
import gc
import json
import math
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb

warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------
# 전제 체크
# ------------------------------------------------------------
required_globals = [
    "PROCESSED_PATH",
    "DATA1_DIR",
    "CELL_SIZE_M",
    "TO_WORK",
    "BRAND_ORDER",
    "SELF_ORDER",
    "read_csv_flexible",
    "normalize_brand",
    "normalize_self",
]
missing = [x for x in required_globals if x not in globals()]
if missing:
    raise RuntimeError(f"먼저 환경설정/1셀을 실행해야 합니다. 누락: {missing}")

# ------------------------------------------------------------
# 보조 함수
# ------------------------------------------------------------
def ql(s: str) -> str:
    return "'" + str(s).replace("'", "''") + "'"

def qi(s: str) -> str:
    return '"' + str(s).replace('"', '""') + '"'

# ------------------------------------------------------------
# 설정
# ------------------------------------------------------------
REGION_ROOT = Path(PROCESSED_PATH) / "additional_data" / "gas_station_prices_by_region"
LAND_GRID_PATH = DATA1_DIR / f"korea_land_grid_{CELL_SIZE_M}m.parquet"
STATION_PANEL_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m.parquet"

WORK_DIR = Path("/content/grid_station_base_work")
RAW_BATCH_DIR = WORK_DIR / "raw_batches"
REGION_AGG_DIR = WORK_DIR / "region_agg"
TMP_ADDITIVE_ALL_PATH = WORK_DIR / "station_additive_all.parquet"

STATION_BATCH_SIZE = 80
DUCKDB_THREADS = 2
CLEANUP_TEMP = True

# ------------------------------------------------------------
# 함수
# ------------------------------------------------------------
def get_station_ids_from_csv(csv_path):
    cols = read_csv_flexible(csv_path, nrows=0).columns.tolist()
    return [c for c in cols if c != "date"]

def wide_subset_to_long(csv_path, value_name, station_ids):
    if not station_ids:
        return pd.DataFrame(columns=["date", "station_id", value_name])

    usecols = ["date"] + list(station_ids)
    wide = read_csv_flexible(csv_path, usecols=usecols)

    wide["date"] = pd.to_datetime(wide["date"], errors="coerce")
    wide = wide.dropna(subset=["date"]).copy()
    wide = (
        wide.sort_values("date")
        .drop_duplicates(subset=["date"], keep="last")
        .copy()
    )

    try:
        stacked = (
            wide.set_index("date")
            .stack(future_stack=True)
            .rename(value_name)
            .reset_index()
            .rename(columns={"level_1": "station_id"})
        )
    except TypeError:
        stacked = (
            wide.set_index("date")
            .stack(dropna=True)
            .rename(value_name)
            .reset_index()
            .rename(columns={"level_1": "station_id"})
        )

    if len(stacked) == 0:
        return pd.DataFrame(columns=["date", "station_id", value_name])

    stacked[value_name] = pd.to_numeric(stacked[value_name], errors="coerce")
    stacked = stacked.dropna(subset=[value_name]).copy()
    stacked = stacked[stacked[value_name] > 0].copy()

    if len(stacked) == 0:
        return pd.DataFrame(columns=["date", "station_id", value_name])

    stacked["station_id"] = stacked["station_id"].astype("string")
    stacked[value_name] = stacked[value_name].astype("float32")
    return stacked

def build_event_table(meta_dict, field):
    rows = []

    for station_id, info in meta_dict.items():
        values = info.get(field, [])
        for item in values:
            if not item:
                continue

            if field == "location":
                if len(item) >= 3:
                    dt, lat, lon = item[0], item[1], item[2]
                    rows.append((station_id, dt, lat, lon))
            else:
                if len(item) >= 2:
                    dt, value = item[0], item[1]
                    rows.append((station_id, dt, value))

    if field == "location":
        out = pd.DataFrame(rows, columns=["station_id", "effective_date", "lat", "lon"])
    else:
        out = pd.DataFrame(rows, columns=["station_id", "effective_date", field])

    if len(out) == 0:
        return out

    out["station_id"] = out["station_id"].astype("string")
    out["effective_date"] = pd.to_datetime(out["effective_date"], errors="coerce")
    out = out.dropna(subset=["effective_date"]).copy()
    out = (
        out.sort_values(["station_id", "effective_date"])
        .drop_duplicates(subset=["station_id", "effective_date"], keep="last")
        .reset_index(drop=True)
    )
    return out

def add_effective_metadata(base_df, event_df):
    if event_df is None or len(event_df) == 0 or len(base_df) == 0:
        return base_df

    left = base_df.sort_values(["date", "station_id"]).copy()
    right = event_df.sort_values(["effective_date", "station_id"]).copy()

    out = pd.merge_asof(
        left,
        right,
        by="station_id",
        left_on="date",
        right_on="effective_date",
        direction="backward",
        allow_exact_matches=True,
    )

    if "effective_date" in out.columns:
        out = out.drop(columns=["effective_date"])

    return out

def assign_grid_cell_xy_from_lonlat(df, lon_col="lon", lat_col="lat", cell_size_m=500):
    out = df.copy()
    x, y = TO_WORK.transform(out[lon_col].to_numpy(), out[lat_col].to_numpy())
    out["cell_x"] = (np.floor(np.asarray(x) / cell_size_m).astype(np.int64) * cell_size_m).astype("int32")
    out["cell_y"] = (np.floor(np.asarray(y) / cell_size_m).astype(np.int64) * cell_size_m).astype("int32")
    return out

def get_station_additive_cols():
    cols = [
        "station_count_total",
        "gasoline_station_count",
        "diesel_station_count",
        "gasoline_price_sum",
        "diesel_price_sum",
    ]
    cols += [f"brand_count__{b}" for b in BRAND_ORDER]
    cols += [f"self_count__{s}" for s in SELF_ORDER]
    for s in SELF_ORDER:
        cols += [
            f"gasoline_price_sum__{s}",
            f"gasoline_price_cnt__{s}",
            f"diesel_price_sum__{s}",
            f"diesel_price_cnt__{s}",
        ]
    return cols

def aggregate_station_day_features(station_day_df):
    if len(station_day_df) == 0:
        return pd.DataFrame()

    work = station_day_df.copy()
    work = work[(work["gasoline_price"].notna()) | (work["diesel_price"].notna())].copy()
    if len(work) == 0:
        return pd.DataFrame()

    work["brand_cat"] = work["brand"].map(normalize_brand)
    work["self_cat"] = work["is_self"].map(normalize_self)

    work["station_count_total"] = 1
    work["gasoline_station_count"] = work["gasoline_price"].notna().astype("int16")
    work["diesel_station_count"] = work["diesel_price"].notna().astype("int16")

    work["gasoline_price_sum"] = work["gasoline_price"].fillna(0).astype("float32")
    work["diesel_price_sum"] = work["diesel_price"].fillna(0).astype("float32")

    for brand in BRAND_ORDER:
        work[f"brand_count__{brand}"] = (work["brand_cat"] == brand).astype("int16")

    for self_label in SELF_ORDER:
        work[f"self_count__{self_label}"] = (work["self_cat"] == self_label).astype("int16")

        gmask = (work["self_cat"] == self_label) & work["gasoline_price"].notna()
        dmask = (work["self_cat"] == self_label) & work["diesel_price"].notna()

        work[f"gasoline_price_sum__{self_label}"] = np.where(
            gmask, work["gasoline_price"], 0
        ).astype("float32")
        work[f"gasoline_price_cnt__{self_label}"] = gmask.astype("int16")

        work[f"diesel_price_sum__{self_label}"] = np.where(
            dmask, work["diesel_price"], 0
        ).astype("float32")
        work[f"diesel_price_cnt__{self_label}"] = dmask.astype("int16")

    additive_cols = get_station_additive_cols()

    out = (
        work.groupby(["date", "cell_x", "cell_y"], observed=True)[additive_cols]
        .sum()
        .reset_index()
    )
    return out

def aggregate_parquets_sum(input_glob, output_parquet, additive_cols):
    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCKDB_THREADS};")

    select_parts = ["date", "cell_x", "cell_y"]
    select_parts += [f"SUM({qi(c)}) AS {qi(c)}" for c in additive_cols]

    sql = f"""
    COPY (
        SELECT
            {", ".join(select_parts)}
        FROM read_parquet({ql(str(input_glob))}, union_by_name=true)
        GROUP BY 1, 2, 3
    ) TO {ql(str(output_parquet))} (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    con.execute(sql)
    con.close()

def process_station_region_to_disk(region_dir, additive_cols):
    region_dir = Path(region_dir)

    gasoline_path = region_dir / "gasoline.csv"
    diesel_path = region_dir / "deisel.csv"
    if not diesel_path.exists():
        diesel_path = region_dir / "diesel.csv"
    metadata_path = region_dir / "metadata__latlon.json"

    if not gasoline_path.exists() or not diesel_path.exists() or not metadata_path.exists():
        raise FileNotFoundError(f"필수 파일 누락: {region_dir}")

    with open(metadata_path, "r", encoding="utf-8") as f:
        meta_dict = json.load(f)

    loc_evt_all = build_event_table(meta_dict, "location")
    brand_evt_all = build_event_table(meta_dict, "brand")
    self_evt_all = build_event_table(meta_dict, "is_self")

    gas_ids = set(get_station_ids_from_csv(gasoline_path))
    diesel_ids = set(get_station_ids_from_csv(diesel_path))
    all_station_ids = sorted(gas_ids | diesel_ids)

    print(f"[region] {region_dir.name} | station 수: {len(all_station_ids):,}")

    batch_files = []
    total_batches = math.ceil(len(all_station_ids) / STATION_BATCH_SIZE) if all_station_ids else 0

    for batch_idx, start in enumerate(range(0, len(all_station_ids), STATION_BATCH_SIZE), start=1):
        batch_ids = all_station_ids[start:start + STATION_BATCH_SIZE]
        batch_id_set = set(batch_ids)

        print(f"  - batch {batch_idx}/{total_batches} | station {len(batch_ids):,}개")

        gas_batch_ids = [sid for sid in batch_ids if sid in gas_ids]
        diesel_batch_ids = [sid for sid in batch_ids if sid in diesel_ids]

        gas_long = wide_subset_to_long(gasoline_path, "gasoline_price", gas_batch_ids)
        diesel_long = wide_subset_to_long(diesel_path, "diesel_price", diesel_batch_ids)

        if len(gas_long) == 0 and len(diesel_long) == 0:
            continue

        station_day = pd.merge(
            gas_long,
            diesel_long,
            on=["date", "station_id"],
            how="outer",
        )

        del gas_long, diesel_long
        gc.collect()

        loc_evt = loc_evt_all.loc[loc_evt_all["station_id"].isin(batch_id_set)].copy() if len(loc_evt_all) else loc_evt_all
        brand_evt = brand_evt_all.loc[brand_evt_all["station_id"].isin(batch_id_set)].copy() if len(brand_evt_all) else brand_evt_all
        self_evt = self_evt_all.loc[self_evt_all["station_id"].isin(batch_id_set)].copy() if len(self_evt_all) else self_evt_all

        station_day = add_effective_metadata(station_day, loc_evt)
        station_day = add_effective_metadata(station_day, brand_evt)
        station_day = add_effective_metadata(station_day, self_evt)

        for c in ["lat", "lon", "brand", "is_self"]:
            if c not in station_day.columns:
                station_day[c] = np.nan

        station_day["lat"] = pd.to_numeric(station_day["lat"], errors="coerce")
        station_day["lon"] = pd.to_numeric(station_day["lon"], errors="coerce")
        station_day = station_day.dropna(subset=["lat", "lon"]).copy()
        station_day = station_day[
            station_day["lon"].between(120, 135) &
            station_day["lat"].between(30, 45)
        ].copy()

        if len(station_day) == 0:
            continue

        station_day = assign_grid_cell_xy_from_lonlat(
            station_day,
            lon_col="lon",
            lat_col="lat",
            cell_size_m=CELL_SIZE_M,
        )

        batch_panel = aggregate_station_day_features(station_day)
        if len(batch_panel) == 0:
            continue

        batch_out = RAW_BATCH_DIR / f"{region_dir.name}__batch_{batch_idx:03d}.parquet"
        batch_panel.to_parquet(batch_out, index=False, compression="zstd")
        batch_files.append(batch_out)

        del station_day, batch_panel, loc_evt, brand_evt, self_evt
        gc.collect()

    if not batch_files:
        print(f"[알림] {region_dir.name}: 유효 관측 없음")
        return None

    region_out = REGION_AGG_DIR / f"{region_dir.name}.parquet"
    aggregate_parquets_sum(
        input_glob=str(RAW_BATCH_DIR / f"{region_dir.name}__batch_*.parquet"),
        output_parquet=str(region_out),
        additive_cols=additive_cols,
    )

    for p in batch_files:
        try:
            p.unlink()
        except Exception:
            pass

    print(f"[완료] {region_dir.name} -> {region_out.name}")
    return region_out

def write_station_base_panel_from_additive(additive_path, land_grid_path, output_path):
    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCKDB_THREADS};")

    select_parts = [
        "a.date",
        f"""COALESCE(
                l.grid_id,
                'G{CELL_SIZE_M}_' || CAST(a.cell_x / {CELL_SIZE_M} AS BIGINT) || '_' || CAST(a.cell_y / {CELL_SIZE_M} AS BIGINT)
            ) AS grid_id""",
        "a.cell_x",
        "a.cell_y",
        "l.center_lon",
        "l.center_lat",

        "CAST(a.station_count_total AS INTEGER) AS station_count_total",
        "CAST(a.gasoline_station_count AS INTEGER) AS gasoline_station_count",
        "CAST(a.diesel_station_count AS INTEGER) AS diesel_station_count",

        "CAST(a.gasoline_price_sum / NULLIF(a.gasoline_station_count, 0) AS FLOAT) AS gasoline_price_mean",
        "CAST(a.diesel_price_sum / NULLIF(a.diesel_station_count, 0) AS FLOAT) AS diesel_price_mean",
    ]

    for brand in BRAND_ORDER:
        c = f"brand_count__{brand}"
        select_parts.append(f"CAST(a.{qi(c)} AS INTEGER) AS {qi(c)}")

    for self_label in SELF_ORDER:
        c = f"self_count__{self_label}"
        select_parts.append(f"CAST(a.{qi(c)} AS INTEGER) AS {qi(c)}")

    for self_label in SELF_ORDER:
        g_sum = f"gasoline_price_sum__{self_label}"
        g_cnt = f"gasoline_price_cnt__{self_label}"
        d_sum = f"diesel_price_sum__{self_label}"
        d_cnt = f"diesel_price_cnt__{self_label}"

        g_out = f"gasoline_price_mean__{self_label}"
        d_out = f"diesel_price_mean__{self_label}"

        select_parts.append(
            f"CAST(a.{qi(g_sum)} / NULLIF(a.{qi(g_cnt)}, 0) AS FLOAT) AS {qi(g_out)}"
        )
        select_parts.append(
            f"CAST(a.{qi(d_sum)} / NULLIF(a.{qi(d_cnt)}, 0) AS FLOAT) AS {qi(d_out)}"
        )

    sql = f"""
    COPY (
        SELECT
            {", ".join(select_parts)}
        FROM read_parquet({ql(str(additive_path))}) a
        LEFT JOIN read_parquet({ql(str(land_grid_path))}) l
        USING (cell_x, cell_y)
    ) TO {ql(str(output_path))} (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    con.execute(sql)
    con.close()

# ------------------------------------------------------------
# 실행
# ------------------------------------------------------------
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

RAW_BATCH_DIR.mkdir(parents=True, exist_ok=True)
REGION_AGG_DIR.mkdir(parents=True, exist_ok=True)

if STATION_PANEL_PATH.exists():
    STATION_PANEL_PATH.unlink()

region_dirs = sorted([p for p in REGION_ROOT.iterdir() if p.is_dir() and not p.name.startswith(".")])
if not region_dirs:
    raise FileNotFoundError(f"region 폴더가 없습니다: {REGION_ROOT}")

additive_cols = get_station_additive_cols()

print("=" * 100)
print("[1] region별 일별-grid 집계")
print(f" - region 수: {len(region_dirs):,}")
print(f" - batch size: {STATION_BATCH_SIZE}")

region_outputs = []
for idx, region_dir in enumerate(region_dirs, start=1):
    print("=" * 100)
    print(f"[{idx}/{len(region_dirs)}] {region_dir.name}")
    out = process_station_region_to_disk(region_dir, additive_cols)
    if out is not None:
        region_outputs.append(out)

if not region_outputs:
    raise RuntimeError("생성된 region 결과가 없습니다.")

print("=" * 100)
print("[2] region 결과 -> 전국 additive parquet")
aggregate_parquets_sum(
    input_glob=str(REGION_AGG_DIR / "*.parquet"),
    output_parquet=str(TMP_ADDITIVE_ALL_PATH),
    additive_cols=additive_cols,
)

print("=" * 100)
print("[3] additive -> 최종 base station panel")
write_station_base_panel_from_additive(
    additive_path=TMP_ADDITIVE_ALL_PATH,
    land_grid_path=LAND_GRID_PATH,
    output_path=STATION_PANEL_PATH,
)

print(f"[저장] {STATION_PANEL_PATH}")

con = duckdb.connect()
summary_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT grid_id) AS unique_grids,
        SUM(CASE WHEN center_lon IS NULL OR center_lat IS NULL THEN 1 ELSE 0 END) AS null_center_rows
    FROM read_parquet({ql(str(STATION_PANEL_PATH))})
""").df()

preview_df = con.execute(f"""
    SELECT *
    FROM read_parquet({ql(str(STATION_PANEL_PATH))})
    LIMIT 10
""").df()
con.close()

display(summary_df)
display(preview_df)

if CLEANUP_TEMP:
    try:
        shutil.rmtree(WORK_DIR)
    except Exception:
        pass

In [ ]:
# ============================================================
# 4단계 A-1셀. 5단계용 세분화 label table 생성
# ------------------------------------------------------------
# 목적:
#   기존 grid_station_daily_panel_500m.parquet는 date-grid 단위라서
#   상표 × 셀프유무 × 유종별 정답을 만들 수 없다.
#
#   이 셀은 원 주유소 일별 가격 + metadata__latlon.json을 다시 사용해서
#   date × grid × brand × self_type × fuel 단위 target label을 만든다.
#
# 산출:
#   DATA1_DIR / "grid_brand_self_fuel_daily_label_500m/"
#
# 주의:
#   이 label table의 target_price_mean은 정답값이다.
#   5단계 feature로 넣으면 안 된다.
# ============================================================

from __future__ import annotations

import os
import gc
import json
import math
import shutil
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. 패키지 준비
# ------------------------------------------------------------
try:
    import duckdb
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

try:
    from pyproj import Transformer
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyproj"])
    from pyproj import Transformer


# ------------------------------------------------------------
# 1. 경로 / 기본 설정
# ------------------------------------------------------------
ROOT_PATH = Path(ROOT_PATH)
PROCESSED_PATH = Path(PROCESSED_PATH)
GRID_DIR = ROOT_PATH / "그리드"
DATA1_DIR = GRID_DIR / "data_1"

CELL_SIZE_M = int(globals().get("CELL_SIZE_M", 500))

REGION_ROOT = PROCESSED_PATH / "additional_data" / "gas_station_prices_by_region"
LAND_GRID_PATH = DATA1_DIR / f"korea_land_grid_{CELL_SIZE_M}m.parquet"

LABEL_OUTPUT_DIR = DATA1_DIR / f"grid_brand_self_fuel_daily_label_{CELL_SIZE_M}m"

WORK_DIR = Path("/content/grid_brand_self_fuel_label_work")
RAW_LABEL_BATCH_DIR = WORK_DIR / "raw_label_batches"
REGION_LABEL_DIR = WORK_DIR / "region_label"

STATION_BATCH_SIZE = int(globals().get("STATION_BATCH_SIZE", 80))
DUCKDB_THREADS = int(globals().get("DUCKDB_THREADS", 4))
CLEANUP_TEMP = True

# 가격 이상치 필터.
# 너무 빡세게 자르면 실제 특이일을 잃을 수 있으므로, 우선 비정상 입력값 제거 목적의 넓은 범위만 둔다.
VALID_PRICE_RULES = {
    "gasoline": {"min": 500.0, "max": 3500.0},
    "diesel":   {"min": 500.0, "max": 4500.0},
}

BRAND_ORDER = list(globals().get("BRAND_ORDER", [
    "SK에너지",
    "GS칼텍스",
    "HD현대오일뱅크",
    "S-OIL",
    "알뜰",
    "NH-OIL",
    "자가상표",
    "기타",
]))

SELF_ORDER = list(globals().get("SELF_ORDER", [
    "셀프",
    "일반",
    "미상",
]))

FUEL_ROWS = [
    {
        "fuel_type": "gasoline",
        "fuel_label": "휘발유",
        "fuel_is_diesel": 0,
        "price_col": "gasoline_price",
    },
    {
        "fuel_type": "diesel",
        "fuel_label": "경유",
        "fuel_is_diesel": 1,
        "price_col": "diesel_price",
    },
]

print("=" * 100)
print("[A-1 설정]")
print("ROOT_PATH       :", ROOT_PATH)
print("PROCESSED_PATH  :", PROCESSED_PATH)
print("DATA1_DIR       :", DATA1_DIR)
print("REGION_ROOT     :", REGION_ROOT)
print("LAND_GRID_PATH  :", LAND_GRID_PATH)
print("LABEL_OUTPUT_DIR:", LABEL_OUTPUT_DIR)
print("CELL_SIZE_M     :", CELL_SIZE_M)
print("BRAND_ORDER     :", BRAND_ORDER)
print("SELF_ORDER      :", SELF_ORDER)
print("=" * 100)


# ------------------------------------------------------------
# 2. 보조 함수
# ------------------------------------------------------------
def ql(s: str | Path) -> str:
    return "'" + str(s).replace("'", "''") + "'"


def qi(s: str) -> str:
    return '"' + str(s).replace('"', '""') + '"'


def nfc(x):
    if x is None:
        return x
    return unicodedata.normalize("NFC", str(x))


def read_csv_flexible_local(path, **kwargs):
    """
    기존 read_csv_flexible이 있으면 그걸 쓰고,
    없으면 cp949/utf-8-sig/euc-kr/utf-8 순서로 읽는다.
    """
    if "read_csv_flexible" in globals():
        return globals()["read_csv_flexible"](path, **kwargs)

    encodings = ["utf-8-sig", "cp949", "euc-kr", "utf-8"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            last_error = e

    raise RuntimeError(f"CSV 읽기 실패: {path} | last_error={last_error}")


def get_station_ids_from_csv_local(csv_path: Path) -> list[str]:
    cols = read_csv_flexible_local(csv_path, nrows=0).columns.tolist()
    return [str(c) for c in cols if str(c) != "date"]


def wide_subset_to_long_local(csv_path: Path, value_name: str, station_ids: list[str]) -> pd.DataFrame:
    """
    wide price CSV:
      date, A0001, A0002, ...
    를 long:
      date, station_id, value_name
    로 변환한다.
    """
    if not station_ids:
        return pd.DataFrame(columns=["date", "station_id", value_name])

    usecols = ["date"] + list(station_ids)
    wide = read_csv_flexible_local(csv_path, usecols=usecols)

    wide["date"] = pd.to_datetime(wide["date"], errors="coerce")
    wide = wide.dropna(subset=["date"]).copy()
    wide = wide.sort_values("date").drop_duplicates(subset=["date"], keep="last").copy()

    try:
        stacked = (
            wide.set_index("date")
            .stack(future_stack=True)
            .rename(value_name)
            .reset_index()
            .rename(columns={"level_1": "station_id"})
        )
    except TypeError:
        stacked = (
            wide.set_index("date")
            .stack(dropna=True)
            .rename(value_name)
            .reset_index()
            .rename(columns={"level_1": "station_id"})
        )

    if len(stacked) == 0:
        return pd.DataFrame(columns=["date", "station_id", value_name])

    stacked["station_id"] = stacked["station_id"].astype("string")
    stacked[value_name] = pd.to_numeric(stacked[value_name], errors="coerce")
    stacked = stacked.dropna(subset=[value_name]).copy()
    stacked = stacked[stacked[value_name] > 0].copy()
    stacked[value_name] = stacked[value_name].astype("float32")

    return stacked


def build_event_table_local(meta_dict: dict, field: str) -> pd.DataFrame:
    """
    metadata__latlon.json 구조:
      info["location"] = [[date, lat, lon], ...]
      info["brand"]    = [[date, brand], ...]
      info["is_self"]  = [[date, is_self], ...]
    """
    rows = []

    for station_id, info in meta_dict.items():
        values = info.get(field, [])

        if not isinstance(values, list):
            continue

        for item in values:
            if not item:
                continue

            if field == "location":
                if isinstance(item, list) and len(item) >= 3:
                    dt, lat, lon = item[0], item[1], item[2]
                    rows.append((station_id, dt, lat, lon))
            else:
                if isinstance(item, list) and len(item) >= 2:
                    dt, value = item[0], item[1]
                    rows.append((station_id, dt, value))

    if field == "location":
        out = pd.DataFrame(rows, columns=["station_id", "effective_date", "lat", "lon"])
    else:
        out = pd.DataFrame(rows, columns=["station_id", "effective_date", field])

    if len(out) == 0:
        return out

    out["station_id"] = out["station_id"].astype("string")
    out["effective_date"] = pd.to_datetime(out["effective_date"], errors="coerce")
    out = out.dropna(subset=["effective_date"]).copy()

    if field == "location":
        out["lat"] = pd.to_numeric(out["lat"], errors="coerce")
        out["lon"] = pd.to_numeric(out["lon"], errors="coerce")

    out = (
        out.sort_values(["station_id", "effective_date"])
        .drop_duplicates(subset=["station_id", "effective_date"], keep="last")
        .reset_index(drop=True)
    )

    return out


def add_effective_metadata_local(base_df: pd.DataFrame, event_df: pd.DataFrame) -> pd.DataFrame:
    if event_df is None or len(event_df) == 0 or len(base_df) == 0:
        return base_df

    left = base_df.sort_values(["date", "station_id"]).copy()
    right = event_df.sort_values(["effective_date", "station_id"]).copy()

    out = pd.merge_asof(
        left,
        right,
        by="station_id",
        left_on="date",
        right_on="effective_date",
        direction="backward",
        allow_exact_matches=True,
    )

    if "effective_date" in out.columns:
        out = out.drop(columns=["effective_date"])

    return out


def normalize_brand_local(x) -> str:
    """
    기존 normalize_brand이 있으면 그걸 우선 사용하되,
    최종 카테고리는 BRAND_ORDER 안으로 맞춘다.
    """
    if "normalize_brand" in globals():
        try:
            y = globals()["normalize_brand"](x)
            y = nfc(y).strip()
            if y in BRAND_ORDER:
                return y
        except Exception:
            pass

    s = "" if pd.isna(x) else nfc(x).strip()
    s0 = s.lower().replace(" ", "").replace("_", "-")

    if s0 in ["sk", "sk에너지", "skenergy", "에스케이", "에스케이에너지"]:
        return "SK에너지"
    if s0 in ["gs", "gs칼텍스", "gscaltex", "지에스", "지에스칼텍스"]:
        return "GS칼텍스"
    if s0 in ["hd", "hd현대오일뱅크", "현대오일뱅크", "현대오일", "hdoilbank"]:
        return "HD현대오일뱅크"
    if s0 in ["s-oil", "soil", "s-oil주식회사", "s오일", "에쓰오일", "에스오일"]:
        return "S-OIL"
    if "알뜰" in s0:
        return "알뜰"
    if s0 in ["nh-oil", "nhoil", "농협", "농협주유소"]:
        return "NH-OIL"
    if "자가" in s0:
        return "자가상표"
    if s in BRAND_ORDER:
        return s

    return "기타"


def normalize_self_local(x) -> str:
    if "normalize_self" in globals():
        try:
            y = globals()["normalize_self"](x)
            y = nfc(y).strip()
            if y in SELF_ORDER:
                return y
        except Exception:
            pass

    s = "" if pd.isna(x) else nfc(x).strip()
    s0 = s.lower().replace(" ", "")

    if s0 in ["셀프", "self", "y", "yes", "1", "true"]:
        return "셀프"
    if s0 in ["일반", "full", "n", "no", "0", "false"]:
        return "일반"
    return "미상"


# 좌표계 변환기.
# 기존 4단계에서 TO_WORK가 있으면 사용하고, 없으면 EPSG:4326 -> EPSG:5179로 생성한다.
if "TO_WORK" in globals():
    TO_WORK_LOCAL = globals()["TO_WORK"]
else:
    TO_WORK_LOCAL = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)


def assign_grid_cell_xy_from_lonlat_local(
    df: pd.DataFrame,
    lon_col: str = "lon",
    lat_col: str = "lat",
    cell_size_m: int = 500,
) -> pd.DataFrame:
    out = df.copy()
    x, y = TO_WORK_LOCAL.transform(out[lon_col].to_numpy(), out[lat_col].to_numpy())
    out["cell_x"] = (
        np.floor(np.asarray(x) / cell_size_m).astype(np.int64) * cell_size_m
    ).astype("int32")
    out["cell_y"] = (
        np.floor(np.asarray(y) / cell_size_m).astype(np.int64) * cell_size_m
    ).astype("int32")
    return out


def valid_price_mask(price: pd.Series, fuel_type: str) -> pd.Series:
    rule = VALID_PRICE_RULES[fuel_type]
    p = pd.to_numeric(price, errors="coerce")
    return p.notna() & p.between(rule["min"], rule["max"])


def station_day_to_brand_self_fuel_label(station_day_df: pd.DataFrame) -> pd.DataFrame:
    """
    station_day_df:
      date, station_id, gasoline_price, diesel_price, lat, lon, brand, is_self, cell_x, cell_y

    return:
      date, cell_x, cell_y, label_brand, label_self_type, fuel_type,
      target_price_sum, target_price_sum_sq, target_station_count, ...
    """
    if len(station_day_df) == 0:
        return pd.DataFrame()

    work = station_day_df.copy()

    work["label_brand"] = work["brand"].map(normalize_brand_local)
    work["label_self_type"] = work["is_self"].map(normalize_self_local)

    parts = []

    for fuel_info in FUEL_ROWS:
        fuel_type = fuel_info["fuel_type"]
        price_col = fuel_info["price_col"]

        if price_col not in work.columns:
            continue

        mask = valid_price_mask(work[price_col], fuel_type)

        if not mask.any():
            continue

        z = work.loc[
            mask,
            [
                "date",
                "station_id",
                "cell_x",
                "cell_y",
                "label_brand",
                "label_self_type",
                price_col,
            ],
        ].copy()

        z = z.rename(columns={price_col: "price"})
        z["price"] = pd.to_numeric(z["price"], errors="coerce").astype("float32")
        z["fuel_type"] = fuel_type
        z["fuel_label"] = fuel_info["fuel_label"]
        z["fuel_is_diesel"] = int(fuel_info["fuel_is_diesel"])

        # region batch 안에서 먼저 집계한다.
        z["target_price_sum"] = z["price"].astype("float64")
        z["target_price_sum_sq"] = (z["price"].astype("float64") ** 2)
        z["target_station_count"] = 1

        g = (
            z.groupby(
                [
                    "date",
                    "cell_x",
                    "cell_y",
                    "label_brand",
                    "label_self_type",
                    "fuel_type",
                    "fuel_label",
                    "fuel_is_diesel",
                ],
                observed=True,
            )
            .agg(
                target_price_sum=("target_price_sum", "sum"),
                target_price_sum_sq=("target_price_sum_sq", "sum"),
                target_station_count=("target_station_count", "sum"),
                target_price_min=("price", "min"),
                target_price_max=("price", "max"),
            )
            .reset_index()
        )

        parts.append(g)

    if not parts:
        return pd.DataFrame()

    out = pd.concat(parts, ignore_index=True)
    return out


def aggregate_region_label_parquets(input_glob: str, output_path: Path):
    """
    batch parquet들을 region 단위로 다시 합산.
    """
    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCKDB_THREADS};")

    sql = f"""
    COPY (
        SELECT
            CAST(date AS DATE) AS date,
            CAST(cell_x AS INTEGER) AS cell_x,
            CAST(cell_y AS INTEGER) AS cell_y,
            CAST(label_brand AS VARCHAR) AS label_brand,
            CAST(label_self_type AS VARCHAR) AS label_self_type,
            CAST(fuel_type AS VARCHAR) AS fuel_type,
            CAST(fuel_label AS VARCHAR) AS fuel_label,
            CAST(fuel_is_diesel AS INTEGER) AS fuel_is_diesel,

            SUM(CAST(target_price_sum AS DOUBLE)) AS target_price_sum,
            SUM(CAST(target_price_sum_sq AS DOUBLE)) AS target_price_sum_sq,
            SUM(CAST(target_station_count AS BIGINT)) AS target_station_count,
            MIN(CAST(target_price_min AS DOUBLE)) AS target_price_min,
            MAX(CAST(target_price_max AS DOUBLE)) AS target_price_max

        FROM read_parquet({ql(input_glob)}, union_by_name=true)
        GROUP BY 1,2,3,4,5,6,7,8
    ) TO {ql(output_path)} (FORMAT PARQUET, COMPRESSION ZSTD)
    """

    con.execute(sql)
    con.close()


def process_station_region_to_label_disk(region_dir: Path) -> Path | None:
    region_dir = Path(region_dir)

    gasoline_path = region_dir / "gasoline.csv"
    diesel_path = region_dir / "deisel.csv"
    if not diesel_path.exists():
        diesel_path = region_dir / "diesel.csv"

    metadata_path = region_dir / "metadata__latlon.json"

    if not gasoline_path.exists() or not diesel_path.exists() or not metadata_path.exists():
        raise FileNotFoundError(
            f"필수 파일 누락: {region_dir}\n"
            f"gasoline={gasoline_path.exists()}, diesel={diesel_path.exists()}, metadata={metadata_path.exists()}"
        )

    with open(metadata_path, "r", encoding="utf-8") as f:
        meta_dict = json.load(f)

    loc_evt_all = build_event_table_local(meta_dict, "location")
    brand_evt_all = build_event_table_local(meta_dict, "brand")
    self_evt_all = build_event_table_local(meta_dict, "is_self")

    gas_ids = set(get_station_ids_from_csv_local(gasoline_path))
    diesel_ids = set(get_station_ids_from_csv_local(diesel_path))
    all_station_ids = sorted(gas_ids | diesel_ids)

    print(f"[region label] {region_dir.name} | station 수: {len(all_station_ids):,}")

    batch_files = []
    total_batches = math.ceil(len(all_station_ids) / STATION_BATCH_SIZE) if all_station_ids else 0

    for batch_idx, start in enumerate(range(0, len(all_station_ids), STATION_BATCH_SIZE), start=1):
        batch_ids = all_station_ids[start:start + STATION_BATCH_SIZE]
        batch_id_set = set(batch_ids)

        print(f"  - batch {batch_idx}/{total_batches} | station {len(batch_ids):,}개")

        gas_batch_ids = [sid for sid in batch_ids if sid in gas_ids]
        diesel_batch_ids = [sid for sid in batch_ids if sid in diesel_ids]

        gas_long = wide_subset_to_long_local(gasoline_path, "gasoline_price", gas_batch_ids)
        diesel_long = wide_subset_to_long_local(diesel_path, "diesel_price", diesel_batch_ids)

        if len(gas_long) == 0 and len(diesel_long) == 0:
            continue

        station_day = pd.merge(
            gas_long,
            diesel_long,
            on=["date", "station_id"],
            how="outer",
        )

        del gas_long, diesel_long
        gc.collect()

        loc_evt = (
            loc_evt_all.loc[loc_evt_all["station_id"].isin(batch_id_set)].copy()
            if len(loc_evt_all)
            else loc_evt_all
        )
        brand_evt = (
            brand_evt_all.loc[brand_evt_all["station_id"].isin(batch_id_set)].copy()
            if len(brand_evt_all)
            else brand_evt_all
        )
        self_evt = (
            self_evt_all.loc[self_evt_all["station_id"].isin(batch_id_set)].copy()
            if len(self_evt_all)
            else self_evt_all
        )

        station_day = add_effective_metadata_local(station_day, loc_evt)
        station_day = add_effective_metadata_local(station_day, brand_evt)
        station_day = add_effective_metadata_local(station_day, self_evt)

        for c in ["lat", "lon", "brand", "is_self"]:
            if c not in station_day.columns:
                station_day[c] = np.nan

        station_day["lat"] = pd.to_numeric(station_day["lat"], errors="coerce")
        station_day["lon"] = pd.to_numeric(station_day["lon"], errors="coerce")

        station_day = station_day.dropna(subset=["lat", "lon"]).copy()
        station_day = station_day[
            station_day["lon"].between(120, 135)
            & station_day["lat"].between(30, 45)
        ].copy()

        if len(station_day) == 0:
            continue

        station_day = assign_grid_cell_xy_from_lonlat_local(
            station_day,
            lon_col="lon",
            lat_col="lat",
            cell_size_m=CELL_SIZE_M,
        )

        batch_label = station_day_to_brand_self_fuel_label(station_day)

        if len(batch_label) == 0:
            continue

        batch_out = RAW_LABEL_BATCH_DIR / f"{region_dir.name}__label_batch_{batch_idx:03d}.parquet"
        batch_label.to_parquet(batch_out, index=False, compression="zstd")
        batch_files.append(batch_out)

        del station_day, batch_label, loc_evt, brand_evt, self_evt
        gc.collect()

    if not batch_files:
        print(f"[알림] {region_dir.name}: 유효 label 관측 없음")
        return None

    region_out = REGION_LABEL_DIR / f"{region_dir.name}.parquet"

    aggregate_region_label_parquets(
        input_glob=str(RAW_LABEL_BATCH_DIR / f"{region_dir.name}__label_batch_*.parquet"),
        output_path=region_out,
    )

    for p in batch_files:
        try:
            p.unlink()
        except Exception:
            pass

    print(f"[완료] {region_dir.name} -> {region_out.name}")
    return region_out


def write_national_brand_self_fuel_label(region_label_glob: str, land_grid_path: Path, output_dir: Path):
    """
    전국 region label들을 합쳐 최종 label directory를 쓴다.
    PARTITION_BY year 사용.
    """
    if output_dir.exists():
        shutil.rmtree(output_dir)

    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCKDB_THREADS};")

    # land_grid에 없는 station patch grid도 있을 수 있으므로 grid_id fallback 생성.
    sql = f"""
    COPY (
        WITH raw AS (
            SELECT
                CAST(date AS DATE) AS date,
                CAST(cell_x AS INTEGER) AS cell_x,
                CAST(cell_y AS INTEGER) AS cell_y,
                CAST(label_brand AS VARCHAR) AS label_brand,
                CAST(label_self_type AS VARCHAR) AS label_self_type,
                CAST(fuel_type AS VARCHAR) AS fuel_type,
                CAST(fuel_label AS VARCHAR) AS fuel_label,
                CAST(fuel_is_diesel AS INTEGER) AS fuel_is_diesel,

                SUM(CAST(target_price_sum AS DOUBLE)) AS target_price_sum,
                SUM(CAST(target_price_sum_sq AS DOUBLE)) AS target_price_sum_sq,
                SUM(CAST(target_station_count AS BIGINT)) AS target_station_count,
                MIN(CAST(target_price_min AS DOUBLE)) AS target_price_min,
                MAX(CAST(target_price_max AS DOUBLE)) AS target_price_max

            FROM read_parquet({ql(region_label_glob)}, union_by_name=true)
            GROUP BY 1,2,3,4,5,6,7,8
        ),
        land AS (
            SELECT
                grid_id,
                CAST(cell_x AS INTEGER) AS cell_x,
                CAST(cell_y AS INTEGER) AS cell_y,
                CAST(center_lon AS DOUBLE) AS center_lon,
                CAST(center_lat AS DOUBLE) AS center_lat
            FROM read_parquet({ql(land_grid_path)})
        )
        SELECT
            EXTRACT(year FROM r.date)::INTEGER AS year,
            r.date,
            COALESCE(
                l.grid_id,
                'G{CELL_SIZE_M}_' || CAST(r.cell_x / {CELL_SIZE_M} AS BIGINT) || '_' || CAST(r.cell_y / {CELL_SIZE_M} AS BIGINT)
            ) AS grid_id,
            r.cell_x,
            r.cell_y,
            l.center_lon,
            l.center_lat,

            r.fuel_type,
            r.fuel_label,
            r.fuel_is_diesel,
            r.label_brand,
            r.label_self_type,

            CAST(r.target_price_sum / NULLIF(r.target_station_count, 0) AS FLOAT) AS target_price_mean,

            CAST(
                CASE
                    WHEN r.target_station_count > 1 THEN
                        SQRT(
                            GREATEST(
                                (r.target_price_sum_sq - r.target_price_sum * r.target_price_sum / r.target_station_count)
                                / NULLIF(r.target_station_count - 1, 0),
                                0.0
                            )
                        )
                    ELSE NULL
                END AS FLOAT
            ) AS target_price_std,

            CAST(r.target_price_min AS FLOAT) AS target_price_min,
            CAST(r.target_price_max AS FLOAT) AS target_price_max,
            CAST(r.target_station_count AS INTEGER) AS target_station_count

        FROM raw r
        LEFT JOIN land l
        USING (cell_x, cell_y)
        WHERE r.target_station_count > 0
    )
    TO {ql(output_dir)}
    (FORMAT PARQUET, COMPRESSION ZSTD, PARTITION_BY (year))
    """

    con.execute(sql)

    summary = con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT date) AS unique_dates,
            COUNT(DISTINCT grid_id) AS unique_grids,
            COUNT(DISTINCT label_brand) AS unique_brands,
            COUNT(DISTINCT label_self_type) AS unique_self_types,
            COUNT(DISTINCT fuel_type) AS unique_fuels,
            MIN(date) AS min_date,
            MAX(date) AS max_date,
            AVG(target_price_mean) AS avg_target_price,
            MIN(target_price_mean) AS min_target_price,
            MAX(target_price_mean) AS max_target_price
        FROM read_parquet({ql(str(output_dir / "**" / "*.parquet"))}, union_by_name=true)
    """).df()

    by_combo = con.execute(f"""
        SELECT
            fuel_type,
            label_brand,
            label_self_type,
            COUNT(*) AS rows,
            SUM(target_station_count) AS station_obs,
            AVG(target_price_mean) AS avg_price
        FROM read_parquet({ql(str(output_dir / "**" / "*.parquet"))}, union_by_name=true)
        GROUP BY 1,2,3
        ORDER BY fuel_type, rows DESC
    """).df()

    con.close()

    print("\n[최종 label summary]")
    print(summary.to_string(index=False))

    print("\n[최종 label by fuel-brand-self]")
    print(by_combo.to_string(index=False))

    return summary, by_combo


# ------------------------------------------------------------
# 3. 실행
# ------------------------------------------------------------
if not REGION_ROOT.exists():
    raise FileNotFoundError(f"REGION_ROOT 없음: {REGION_ROOT}")

if not LAND_GRID_PATH.exists():
    raise FileNotFoundError(f"LAND_GRID_PATH 없음: {LAND_GRID_PATH}")

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

RAW_LABEL_BATCH_DIR.mkdir(parents=True, exist_ok=True)
REGION_LABEL_DIR.mkdir(parents=True, exist_ok=True)

region_dirs = sorted([p for p in REGION_ROOT.iterdir() if p.is_dir() and not p.name.startswith(".")])

if not region_dirs:
    raise FileNotFoundError(f"region 폴더가 없습니다: {REGION_ROOT}")

print("=" * 100)
print("[A-1 실행] region별 brand-self-fuel label 생성")
print(f" - region 수: {len(region_dirs):,}")
print(f" - batch size: {STATION_BATCH_SIZE}")
print("=" * 100)

region_outputs = []

for idx, region_dir in enumerate(region_dirs, start=1):
    print("=" * 100)
    print(f"[{idx}/{len(region_dirs)}] {region_dir.name}")
    out = process_station_region_to_label_disk(region_dir)
    if out is not None:
        region_outputs.append(out)

if not region_outputs:
    raise RuntimeError("생성된 region label 결과가 없습니다.")

print("=" * 100)
print("[A-1 실행] region label -> 전국 최종 label directory")
label_summary_df, label_by_combo_df = write_national_brand_self_fuel_label(
    region_label_glob=str(REGION_LABEL_DIR / "*.parquet"),
    land_grid_path=LAND_GRID_PATH,
    output_dir=LABEL_OUTPUT_DIR,
)

print(f"\n[저장 완료] {LABEL_OUTPUT_DIR}")

if CLEANUP_TEMP:
    try:
        shutil.rmtree(WORK_DIR)
        print(f"[임시폴더 삭제] {WORK_DIR}")
    except Exception as e:
        print(f"[임시폴더 삭제 실패] {WORK_DIR} | {e}")

#### 주유소 데이터 시각화

In [ ]:
# ============================================================
# A 시각화 셀
# - 특정 날짜 1개
# - 2개 그림:
#   1) station_count_total
#   2) gasoline_price_mean
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

def ql(s: str) -> str:
    return "'" + str(s).replace("'", "''") + "'"

PANEL_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m.parquet"
CHECK_DATE = None   # 예: "2024-01-15", None이면 station_total 가장 큰 날짜 자동 선택

def plot_grid_map(df, value_col, title, log1p=False, robust=False, cmap="YlOrRd"):
    work = df[["center_lon", "center_lat", value_col]].dropna().copy()
    if len(work) == 0:
        print(f"[plot skip] {value_col}: 데이터 없음")
        return

    values = work[value_col].astype(float).to_numpy()
    if log1p:
        values = np.log1p(np.maximum(values, 0))

    vmin, vmax = None, None
    finite_vals = values[np.isfinite(values)]
    if robust and len(finite_vals) >= 20:
        vmin, vmax = np.nanpercentile(finite_vals, [1, 99])
        if np.isclose(vmin, vmax):
            vmin, vmax = None, None

    n = len(work)
    if n >= 250000:
        point_size = 1.0
    elif n >= 120000:
        point_size = 2.0
    elif n >= 50000:
        point_size = 3.5
    else:
        point_size = 6.0

    work = work.assign(_plot_value=values).sort_values("_plot_value")

    fig, ax = plt.subplots(figsize=(7.2, 9.2))
    sc = ax.scatter(
        work["center_lon"],
        work["center_lat"],
        c=work["_plot_value"],
        s=point_size,
        marker="s",
        linewidths=0,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        alpha=0.95,
        rasterized=True,
    )
    ax.set_xlim(124.0, 132.2)
    ax.set_ylim(33.0, 39.7)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title)
    ax.set_aspect(1 / np.cos(np.deg2rad(36.0)))
    plt.colorbar(sc, ax=ax, shrink=0.85)
    plt.tight_layout()
    plt.show()

con = duckdb.connect()

if CHECK_DATE is None:
    top_date_df = con.execute(f"""
        SELECT
            CAST(date AS DATE) AS dt,
            SUM(station_count_total) AS station_total
        FROM read_parquet({ql(str(PANEL_PATH))})
        GROUP BY 1
        ORDER BY station_total DESC, dt DESC
        LIMIT 1
    """).df()
    SELECTED_DATE = str(pd.to_datetime(top_date_df["dt"].iloc[0]).date())
else:
    SELECTED_DATE = str(pd.to_datetime(CHECK_DATE).date())

plot_df = con.execute(f"""
    SELECT
        date,
        grid_id,
        center_lon,
        center_lat,
        station_count_total,
        gasoline_price_mean
    FROM read_parquet({ql(str(PANEL_PATH))})
    WHERE date = CAST({ql(SELECTED_DATE)} AS DATE)
""").df()

con.close()

print(f"[선택 날짜] {SELECTED_DATE}")
display(
    plot_df[["station_count_total", "gasoline_price_mean"]]
    .describe()
    .T
)

plot_grid_map(
    plot_df,
    value_col="station_count_total",
    title=f"{SELECTED_DATE} | Station Count (log1p)",
    log1p=True,
    robust=False,
    cmap="YlOrRd",
)

plot_grid_map(
    plot_df,
    value_col="gasoline_price_mean",
    title=f"{SELECTED_DATE} | Gasoline Mean Price",
    log1p=False,
    robust=True,
    cmap="viridis",
)

#### 주유소 영향력 데이터 생성

In [ ]:
# ============================================================
# B 셀. 주유소 영향력만 따로 계산해서 붙이기
# - 입력: grid_station_daily_panel_500m.parquet
# - 출력: grid_station_daily_panel_500m_plus_station_influence.parquet
# - 업데이트 버전:
#   연도별 -> 월별 partition으로 RAM 안정성 개선
# ============================================================

import gc
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb

from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=FutureWarning)

def ql(s: str) -> str:
    return "'" + str(s).replace("'", "''") + "'"

CELL_SIZE_M = globals().get("CELL_SIZE_M", 500)

BASE_PANEL_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m.parquet"
OUTPUT_PANEL_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m_plus_station_influence.parquet"

WORK_DIR = Path("/content/station_influence_work")
KEYS_BY_MONTH_DIR = WORK_DIR / "keys_by_month"
INFL_BY_MONTH_DIR = WORK_DIR / "influence_by_month"

DUCKDB_THREADS = 2
STATION_INFLUENCE_BAND_KM = 3.0
STATION_INFLUENCE_CUTOFF_KM = 15.0

def compute_station_neighbor_influence_observed(cell_x, cell_y, counts, band_km=3.0, cutoff_km=15.0):
    """
    관측된 grid row들에 대해서만 influence 계산
    - 자기 grid 제외
    - 같은 날짜 다른 grid들의 station_count_total 감쇠합
    """
    n = len(counts)
    if n <= 1:
        return np.zeros(n, dtype="float32")

    coords = np.column_stack([
        cell_x.astype("float64") + CELL_SIZE_M / 2.0,
        cell_y.astype("float64") + CELL_SIZE_M / 2.0,
    ])

    band_m = band_km * 1000.0
    cutoff_m = cutoff_km * 1000.0

    tree = cKDTree(coords)
    coo = tree.sparse_distance_matrix(tree, cutoff_m, output_type="coo_matrix")

    if coo.nnz == 0:
        return np.zeros(n, dtype="float32")

    # 자기 자신 제외
    mask = coo.row != coo.col
    if mask.sum() == 0:
        return np.zeros(n, dtype="float32")

    weights = np.exp(-coo.data[mask] / band_m) * counts[coo.col[mask]]
    influence = np.bincount(coo.row[mask], weights=weights, minlength=n)

    return influence.astype("float32")

def build_station_influence_month_files(base_panel_path, keys_by_month_dir, infl_by_month_dir):
    keys_by_month_dir = Path(keys_by_month_dir)
    infl_by_month_dir = Path(infl_by_month_dir)

    if keys_by_month_dir.exists():
        shutil.rmtree(keys_by_month_dir)
    if infl_by_month_dir.exists():
        shutil.rmtree(infl_by_month_dir)

    keys_by_month_dir.mkdir(parents=True, exist_ok=True)
    infl_by_month_dir.mkdir(parents=True, exist_ok=True)

    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCKDB_THREADS};")

    con.execute(f"""
        COPY (
            SELECT
                date,
                cell_x,
                cell_y,
                station_count_total,
                EXTRACT(year FROM date)::INTEGER AS year,
                EXTRACT(month FROM date)::INTEGER AS month
            FROM read_parquet({ql(str(base_panel_path))})
        ) TO {ql(str(keys_by_month_dir))}
        (FORMAT PARQUET, PARTITION_BY (year, month), COMPRESSION ZSTD)
    """)
    con.close()

    month_dirs = sorted(keys_by_month_dir.glob("year=*/month=*"))
    if not month_dirs:
        raise RuntimeError("월별 key parquet가 생성되지 않았습니다.")

    for month_dir in month_dirs:
        year_name = month_dir.parent.name.split("=")[-1]
        month_name = month_dir.name.split("=")[-1]

        print(f"[station influence] year={year_name}, month={month_name}")

        df_month = pd.read_parquet(month_dir)
        if len(df_month) == 0:
            continue

        parts = []
        total_dates = df_month["date"].nunique()

        for idx, (dt, g) in enumerate(df_month.groupby("date", sort=True), start=1):
            infl = compute_station_neighbor_influence_observed(
                cell_x=g["cell_x"].to_numpy(),
                cell_y=g["cell_y"].to_numpy(),
                counts=g["station_count_total"].to_numpy(dtype="float64"),
                band_km=STATION_INFLUENCE_BAND_KM,
                cutoff_km=STATION_INFLUENCE_CUTOFF_KM,
            )

            part = g[["date", "cell_x", "cell_y"]].copy()
            part["station_neighbor_influence"] = infl
            parts.append(part)

            if idx % 20 == 0 or idx == total_dates:
                print(f"  - {idx:,}/{total_dates:,} dates")

        out_path = infl_by_month_dir / f"station_influence_{int(year_name):04d}_{int(month_name):02d}.parquet"
        pd.concat(parts, ignore_index=True).to_parquet(
            out_path,
            index=False,
            compression="zstd",
        )

        del df_month, parts
        gc.collect()

def attach_station_influence_to_panel(base_panel_path, infl_glob, output_path):
    base_panel_path = Path(base_panel_path)
    output_path = Path(output_path)

    if output_path.exists():
        output_path.unlink()

    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCKDB_THREADS};")

    schema_df = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet({ql(str(base_panel_path))})"
    ).df()
    existing_cols = schema_df["column_name"].tolist()

    if "station_neighbor_influence" in existing_cols:
        base_sql = (
            f"SELECT * EXCLUDE (station_neighbor_influence) "
            f"FROM read_parquet({ql(str(base_panel_path))})"
        )
    else:
        base_sql = f"SELECT * FROM read_parquet({ql(str(base_panel_path))})"

    sql = f"""
    COPY (
        WITH base AS (
            {base_sql}
        ),
        infl AS (
            SELECT *
            FROM read_parquet({ql(str(infl_glob))}, union_by_name=true)
        )
        SELECT
            b.*,
            CAST(COALESCE(i.station_neighbor_influence, 0.0) AS FLOAT) AS station_neighbor_influence
        FROM base b
        LEFT JOIN infl i
        USING (date, cell_x, cell_y)
    ) TO {ql(str(output_path))} (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    con.execute(sql)
    con.close()

# ------------------------------------------------------------
# 실행
# ------------------------------------------------------------
if not BASE_PANEL_PATH.exists():
    raise FileNotFoundError(f"기본 station panel이 없습니다: {BASE_PANEL_PATH}")

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

KEYS_BY_MONTH_DIR.mkdir(parents=True, exist_ok=True)
INFL_BY_MONTH_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("[1] 월별 key parquet 생성 + influence 계산")
build_station_influence_month_files(
    base_panel_path=BASE_PANEL_PATH,
    keys_by_month_dir=KEYS_BY_MONTH_DIR,
    infl_by_month_dir=INFL_BY_MONTH_DIR,
)

print("=" * 100)
print("[2] influence -> panel 병합")
attach_station_influence_to_panel(
    base_panel_path=BASE_PANEL_PATH,
    infl_glob=str(INFL_BY_MONTH_DIR / "*.parquet"),
    output_path=OUTPUT_PANEL_PATH,
)

print(f"[저장] {OUTPUT_PANEL_PATH}")

con = duckdb.connect()
summary_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT grid_id) AS unique_grids,
        AVG(station_neighbor_influence) AS avg_station_neighbor_influence
    FROM read_parquet({ql(str(OUTPUT_PANEL_PATH))})
""").df()

preview_df = con.execute(f"""
    SELECT
        date,
        grid_id,
        station_count_total,
        gasoline_price_mean,
        station_neighbor_influence
    FROM read_parquet({ql(str(OUTPUT_PANEL_PATH))})
    LIMIT 10
""").df()
con.close()

display(summary_df)
display(preview_df)

try:
    shutil.rmtree(WORK_DIR)
except Exception:
    pass

#### 주유소 영향력 시각화

In [ ]:
# ============================================================
# B 시각화 셀
# - 특정 날짜 1개
# - full land grid 기준으로 station influence를 on-the-fly 계산
# - 그림 1개:
#   station_neighbor_influence_fullgrid
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree

def ql(s: str) -> str:
    return "'" + str(s).replace("'", "''") + "'"

CELL_SIZE_M = globals().get("CELL_SIZE_M", 500)

LAND_GRID_PATH = DATA1_DIR / f"korea_land_grid_{CELL_SIZE_M}m.parquet"
PANEL_PLUS_STATION_INFL_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m_plus_station_influence.parquet"
BASE_PANEL_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m.parquet"

if PANEL_PLUS_STATION_INFL_PATH.exists():
    SOURCE_PANEL_PATH = PANEL_PLUS_STATION_INFL_PATH
else:
    SOURCE_PANEL_PATH = BASE_PANEL_PATH

CHECK_DATE = None   # 예: "2024-01-15", None이면 station_total 가장 큰 날짜 자동 선택
STATION_INFLUENCE_BAND_KM = 3.0
STATION_INFLUENCE_CUTOFF_KM = 15.0
INFL_CHUNK_SIZE = 20000

def auto_point_size(n):
    if n >= 250000:
        return 1.0
    elif n >= 120000:
        return 2.0
    elif n >= 50000:
        return 3.5
    else:
        return 6.0

def plot_grid_map(df, value_col, title, log1p=False, robust=False, cmap="plasma"):
    work = df[["center_lon", "center_lat", value_col]].dropna().copy()
    if len(work) == 0:
        print(f"[plot skip] {value_col}: 데이터 없음")
        return

    values = work[value_col].astype(float).to_numpy()
    if log1p:
        values = np.log1p(np.maximum(values, 0))

    vmin, vmax = None, None
    finite_vals = values[np.isfinite(values)]
    if robust and len(finite_vals) >= 20:
        vmin, vmax = np.nanpercentile(finite_vals, [1, 99])
        if np.isclose(vmin, vmax):
            vmin, vmax = None, None

    work = work.assign(_plot_value=values).sort_values("_plot_value")

    fig, ax = plt.subplots(figsize=(7.2, 9.2))
    sc = ax.scatter(
        work["center_lon"],
        work["center_lat"],
        c=work["_plot_value"],
        s=auto_point_size(len(work)),
        marker="s",
        linewidths=0,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        alpha=0.95,
        rasterized=True,
    )
    ax.set_xlim(124.0, 132.2)
    ax.set_ylim(33.0, 39.7)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title)
    ax.set_aspect(1 / np.cos(np.deg2rad(36.0)))
    plt.colorbar(sc, ax=ax, shrink=0.85)
    plt.tight_layout()
    plt.show()

def compute_fullgrid_station_influence(
    target_x,
    target_y,
    source_x,
    source_y,
    source_weights,
    band_km=3.0,
    cutoff_km=15.0,
    chunk_size=20000,
):
    """
    특정 날짜에 대해 full land grid 전체를 target으로 influence 계산
    - 자기 grid 제외
    """
    target_coords = np.column_stack([target_x.astype("float64"), target_y.astype("float64")])
    source_coords = np.column_stack([source_x.astype("float64"), source_y.astype("float64")])

    if len(source_coords) == 0:
        return np.zeros(len(target_coords), dtype="float32")

    band_m = band_km * 1000.0
    cutoff_m = cutoff_km * 1000.0

    source_tree = cKDTree(source_coords)
    out = np.zeros(len(target_coords), dtype="float32")

    for start in range(0, len(target_coords), chunk_size):
        end = min(start + chunk_size, len(target_coords))
        tc = target_coords[start:end]

        neighbor_lists = source_tree.query_ball_point(tc, r=cutoff_m)
        vals = np.zeros(len(tc), dtype="float64")

        for i, idxs in enumerate(neighbor_lists):
            if not idxs:
                continue

            src = source_coords[idxs]
            d = np.hypot(src[:, 0] - tc[i, 0], src[:, 1] - tc[i, 1])

            # 자기 자신 grid 제외
            same = (src[:, 0] == tc[i, 0]) & (src[:, 1] == tc[i, 1])
            if same.any():
                d = d[~same]
                sw = source_weights[idxs][~same]
            else:
                sw = source_weights[idxs]

            if len(d) > 0:
                vals[i] = np.sum(np.exp(-d / band_m) * sw)

        out[start:end] = vals.astype("float32")

    return out

# ------------------------------------------------------------
# 날짜 선택
# ------------------------------------------------------------
con = duckdb.connect()

if CHECK_DATE is None:
    top_date_df = con.execute(f"""
        SELECT
            CAST(date AS DATE) AS dt,
            SUM(station_count_total) AS station_total
        FROM read_parquet({ql(str(SOURCE_PANEL_PATH))})
        GROUP BY 1
        ORDER BY station_total DESC, dt DESC
        LIMIT 1
    """).df()
    SELECTED_DATE = str(pd.to_datetime(top_date_df["dt"].iloc[0]).date())
else:
    SELECTED_DATE = str(pd.to_datetime(CHECK_DATE).date())

observed_df = con.execute(f"""
    SELECT
        date,
        cell_x,
        cell_y,
        station_count_total
    FROM read_parquet({ql(str(SOURCE_PANEL_PATH))})
    WHERE date = CAST({ql(SELECTED_DATE)} AS DATE)
""").df()
con.close()

if len(observed_df) == 0:
    raise RuntimeError(f"{SELECTED_DATE} 날짜의 주유소 데이터가 없습니다.")

land_grid_df = pd.read_parquet(
    LAND_GRID_PATH,
    columns=["grid_id", "cell_x", "cell_y", "center_lon", "center_lat"]
)

target_x = land_grid_df["cell_x"].to_numpy(dtype="float64") + CELL_SIZE_M / 2.0
target_y = land_grid_df["cell_y"].to_numpy(dtype="float64") + CELL_SIZE_M / 2.0

source_x = observed_df["cell_x"].to_numpy(dtype="float64") + CELL_SIZE_M / 2.0
source_y = observed_df["cell_y"].to_numpy(dtype="float64") + CELL_SIZE_M / 2.0
source_w = observed_df["station_count_total"].to_numpy(dtype="float64")

land_grid_df = land_grid_df.copy()
land_grid_df["station_neighbor_influence_fullgrid"] = compute_fullgrid_station_influence(
    target_x=target_x,
    target_y=target_y,
    source_x=source_x,
    source_y=source_y,
    source_weights=source_w,
    band_km=STATION_INFLUENCE_BAND_KM,
    cutoff_km=STATION_INFLUENCE_CUTOFF_KM,
    chunk_size=INFL_CHUNK_SIZE,
)

print(f"[선택 날짜] {SELECTED_DATE}")
display(
    land_grid_df[["station_neighbor_influence_fullgrid"]]
    .describe()
    .T
)

plot_grid_map(
    land_grid_df,
    value_col="station_neighbor_influence_fullgrid",
    title=f"{SELECTED_DATE} | Station Neighbor Influence (full land grid, log1p)",
    log1p=True,
    robust=True,
    cmap="plasma",
)

#### 저유소/대리점/공장 데이터 넣기

In [ ]:
# ============================================================
# 4셀. 저유소/대리점/공장 데이터 넣기
# - full land grid 기준 static count / static influence 계산
# - 그 값을 station daily panel의 모든 날짜 행에 동일하게 붙임
# 결과:
#   1) facility_effect_land_grid_static_500m.parquet
#   2) grid_station_daily_panel_500m_plus_facility.parquet
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

FACILITY_CSV_PATH = Path(PROCESSED_PATH) / "additional_data" / "1 facility_location_data_final.csv"
LAND_GRID_PATH = DATA1_DIR / f"korea_land_grid_{CELL_SIZE_M}m.parquet"
INPUT_PANEL_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m.parquet"

FACILITY_STATIC_PATH = DATA1_DIR / f"facility_effect_land_grid_static_{CELL_SIZE_M}m.parquet"
PANEL_PLUS_FACILITY_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m_plus_facility.parquet"

# 영향력 파라미터
FACILITY_DECAY_CONFIG = {
    "storage": {"band_km": 20.0, "cutoff_km": 60.0},
    "agency":  {"band_km": 10.0, "cutoff_km": 30.0},
    "factory": {"band_km": 35.0, "cutoff_km": 105.0},
}
FACILITY_CHUNK_SIZE = 5000

def load_facility_points(facility_csv_path, cell_size_m=500):
    raw = read_csv_flexible(facility_csv_path)

    expected = {"상표", "대상", "경도", "위도"}
    if not expected.issubset(raw.columns):
        raise ValueError(f"시설 파일 컬럼이 예상과 다릅니다. 현재 컬럼: {raw.columns.tolist()}")

    pts = raw.rename(columns={
        "상표": "brand",
        "대상": "facility_type_raw",
        "경도": "lon",
        "위도": "lat",
    }).copy()

    pts["lon"] = pd.to_numeric(pts["lon"], errors="coerce")
    pts["lat"] = pd.to_numeric(pts["lat"], errors="coerce")
    pts = pts.dropna(subset=["lon", "lat", "facility_type_raw"]).copy()

    pts = pts[
        pts["lon"].between(120, 135) &
        pts["lat"].between(30, 45)
    ].copy()

    pts["facility_type"] = pts["facility_type_raw"].map(normalize_facility_type)
    pts = pts.dropna(subset=["facility_type"]).copy()

    x, y = TO_WORK.transform(pts["lon"].to_numpy(), pts["lat"].to_numpy())
    pts["x_m"] = np.asarray(x, dtype="float64")
    pts["y_m"] = np.asarray(y, dtype="float64")
    pts["cell_x"] = (np.floor(pts["x_m"] / cell_size_m).astype(np.int64) * cell_size_m).astype("int32")
    pts["cell_y"] = (np.floor(pts["y_m"] / cell_size_m).astype(np.int64) * cell_size_m).astype("int32")

    return pts

def build_facility_counts_on_land_grid(land_grid_df, facility_points_df):
    tmp = facility_points_df.copy()
    tmp["facility_count_total"] = 1
    tmp["facility_storage_count"] = (tmp["facility_type"] == "storage").astype("int16")
    tmp["facility_factory_count"] = (tmp["facility_type"] == "factory").astype("int16")
    tmp["facility_agency_count"] = (tmp["facility_type"] == "agency").astype("int16")

    cnt = (
        tmp.groupby(["cell_x", "cell_y"], observed=True)
        .agg(
            facility_count_total=("facility_count_total", "sum"),
            facility_storage_count=("facility_storage_count", "sum"),
            facility_factory_count=("facility_factory_count", "sum"),
            facility_agency_count=("facility_agency_count", "sum"),
        )
        .reset_index()
    )

    out = land_grid_df[["cell_x", "cell_y"]].merge(
        cnt,
        on=["cell_x", "cell_y"],
        how="left",
    )

    for c in [
        "facility_count_total",
        "facility_storage_count",
        "facility_factory_count",
        "facility_agency_count",
    ]:
        out[c] = out[c].fillna(0).astype("int32")

    return out

def compute_decay_sum_raw(target_xy, source_xy, band_km, cutoff_km=None, chunk_size=5000):
    n_target = target_xy.shape[0]
    raw_score = np.zeros(n_target, dtype="float64")

    if source_xy.shape[0] == 0:
        return raw_score.astype("float32")

    band_m = float(band_km) * 1000.0
    cutoff_m = None if cutoff_km is None else float(cutoff_km) * 1000.0

    sx = source_xy[:, 0][None, :]
    sy = source_xy[:, 1][None, :]

    for start in range(0, n_target, chunk_size):
        end = min(start + chunk_size, n_target)

        tx = target_xy[start:end, 0][:, None]
        ty = target_xy[start:end, 1][:, None]

        d = np.hypot(tx - sx, ty - sy)
        w = np.exp(-d / band_m)

        if cutoff_m is not None:
            w[d > cutoff_m] = 0.0

        raw_score[start:end] = w.sum(axis=1)

    return raw_score.astype("float32")

def build_facility_static_on_land_grid(land_grid_df, facility_points_df, decay_config):
    out = land_grid_df.copy()

    count_df = build_facility_counts_on_land_grid(out, facility_points_df)
    out = out.merge(count_df, on=["cell_x", "cell_y"], how="left")

    target_xy = out[["center_x", "center_y"]].to_numpy(dtype="float64")

    for facility_type, prefix in [
        ("storage", "storage"),
        ("agency", "agency"),
        ("factory", "factory"),
    ]:
        src = facility_points_df.loc[
            facility_points_df["facility_type"] == facility_type,
            ["x_m", "y_m"]
        ].to_numpy(dtype="float64")

        out[f"{prefix}_influence"] = compute_decay_sum_raw(
            target_xy=target_xy,
            source_xy=src,
            band_km=decay_config[facility_type]["band_km"],
            cutoff_km=decay_config[facility_type]["cutoff_km"],
            chunk_size=FACILITY_CHUNK_SIZE,
        )

    return out[[
        "grid_id",
        "cell_x", "cell_y",
        "center_lon", "center_lat",
        "facility_count_total",
        "facility_storage_count",
        "facility_factory_count",
        "facility_agency_count",
        "storage_influence",
        "agency_influence",
        "factory_influence",
    ]]

def attach_facility_static_to_panel(input_panel_path, static_facility_path, output_panel_path):
    if Path(output_panel_path).exists():
        Path(output_panel_path).unlink()

    con = duckdb.connect()

    schema_df = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet({ql(str(input_panel_path))})"
    ).df()
    existing_cols = schema_df["column_name"].tolist()

    facility_cols = [
        "facility_count_total",
        "facility_storage_count",
        "facility_factory_count",
        "facility_agency_count",
        "storage_influence",
        "agency_influence",
        "factory_influence",
    ]
    overlap_cols = [c for c in facility_cols if c in existing_cols]

    if overlap_cols:
        panel_sql = (
            f"SELECT * EXCLUDE ({', '.join(overlap_cols)}) "
            f"FROM read_parquet({ql(str(input_panel_path))})"
        )
    else:
        panel_sql = f"SELECT * FROM read_parquet({ql(str(input_panel_path))})"

    sql = f"""
    COPY (
        WITH panel AS (
            {panel_sql}
        ),
        fac AS (
            SELECT
                cell_x,
                cell_y,
                facility_count_total,
                facility_storage_count,
                facility_factory_count,
                facility_agency_count,
                storage_influence,
                agency_influence,
                factory_influence
            FROM read_parquet({ql(str(static_facility_path))})
        )
        SELECT
            p.*,
            CAST(COALESCE(f.facility_count_total, 0) AS INTEGER) AS facility_count_total,
            CAST(COALESCE(f.facility_storage_count, 0) AS INTEGER) AS facility_storage_count,
            CAST(COALESCE(f.facility_factory_count, 0) AS INTEGER) AS facility_factory_count,
            CAST(COALESCE(f.facility_agency_count, 0) AS INTEGER) AS facility_agency_count,
            CAST(COALESCE(f.storage_influence, 0.0) AS FLOAT) AS storage_influence,
            CAST(COALESCE(f.agency_influence, 0.0) AS FLOAT) AS agency_influence,
            CAST(COALESCE(f.factory_influence, 0.0) AS FLOAT) AS factory_influence
        FROM panel p
        LEFT JOIN fac f
        USING (cell_x, cell_y)
    ) TO {ql(str(output_panel_path))} (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    con.execute(sql)
    con.close()

# ------------------------------------------------------------
# 실행
# ------------------------------------------------------------
print("=" * 100)
print("[1] land grid 로드")
land_grid_df = pd.read_parquet(
    LAND_GRID_PATH,
    columns=[
        "grid_id",
        "cell_x", "cell_y",
        "center_x", "center_y",
        "center_lon", "center_lat",
    ],
)
print(f" - land grid 수: {len(land_grid_df):,}")

print("=" * 100)
print("[2] 시설 포인트 로드")
facility_points_df = load_facility_points(FACILITY_CSV_PATH, cell_size_m=CELL_SIZE_M)
print(
    facility_points_df["facility_type"]
    .value_counts()
    .rename(index={"storage": "저유소", "agency": "대리점", "factory": "공장"})
    .to_frame("count")
)

print("=" * 100)
print("[3] full land grid 기준 static facility count / influence 계산")
facility_static_df = build_facility_static_on_land_grid(
    land_grid_df=land_grid_df,
    facility_points_df=facility_points_df,
    decay_config=FACILITY_DECAY_CONFIG,
)
facility_static_df.to_parquet(FACILITY_STATIC_PATH, index=False, compression="zstd")
print(f"[저장] {FACILITY_STATIC_PATH}")

display(
    facility_static_df[[
        "facility_storage_count",
        "facility_agency_count",
        "facility_factory_count",
        "storage_influence",
        "agency_influence",
        "factory_influence",
    ]].describe().T
)

print("=" * 100)
print("[4] station panel에 static facility 값 붙이기")
attach_facility_static_to_panel(
    input_panel_path=INPUT_PANEL_PATH,
    static_facility_path=FACILITY_STATIC_PATH,
    output_panel_path=PANEL_PLUS_FACILITY_PATH,
)
print(f"[저장] {PANEL_PLUS_FACILITY_PATH}")

con = duckdb.connect()
summary_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT grid_id) AS unique_grids
    FROM read_parquet({ql(str(PANEL_PLUS_FACILITY_PATH))})
""").df()

preview_df = con.execute(f"""
    SELECT
        date,
        grid_id,
        facility_storage_count,
        facility_agency_count,
        facility_factory_count,
        storage_influence,
        agency_influence,
        factory_influence
    FROM read_parquet({ql(str(PANEL_PLUS_FACILITY_PATH))})
    LIMIT 10
""").df()
con.close()

display(summary_df)
display(preview_df)

#### 시설 데이터 시각화

In [ ]:
# ============================================================
# 5셀. 시설 데이터 시각화
# - static file 기준
# - 6개 그림:
#   1) storage count
#   2) agency count
#   3) factory count
#   4) storage influence
#   5) agency influence
#   6) factory influence
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FACILITY_STATIC_PATH = DATA1_DIR / f"facility_effect_land_grid_static_{CELL_SIZE_M}m.parquet"

def auto_point_size(n):
    if n >= 250000:
        return 1.0
    elif n >= 120000:
        return 2.0
    elif n >= 50000:
        return 3.5
    else:
        return 6.0

def plot_static_grid(df, value_col, title, log1p=False, robust=False, cmap="YlOrRd"):
    work = df[["center_lon", "center_lat", value_col]].dropna().copy()
    if len(work) == 0:
        print(f"[plot skip] {value_col}: 데이터 없음")
        return

    values = work[value_col].astype(float).to_numpy()
    if log1p:
        values = np.log1p(np.maximum(values, 0))

    vmin, vmax = None, None
    finite_vals = values[np.isfinite(values)]
    if robust and len(finite_vals) >= 20:
        vmin, vmax = np.nanpercentile(finite_vals, [1, 99])
        if np.isclose(vmin, vmax):
            vmin, vmax = None, None

    point_size = auto_point_size(len(work))
    work = work.assign(_plot_value=values).sort_values("_plot_value")

    fig, ax = plt.subplots(figsize=(7.2, 9.2))
    sc = ax.scatter(
        work["center_lon"],
        work["center_lat"],
        c=work["_plot_value"],
        s=point_size,
        marker="s",
        linewidths=0,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        alpha=0.95,
        rasterized=True,
    )
    ax.set_xlim(124.0, 132.2)
    ax.set_ylim(33.0, 39.7)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title)
    ax.set_aspect(1 / np.cos(np.deg2rad(36.0)))
    plt.colorbar(sc, ax=ax, shrink=0.85)
    plt.tight_layout()
    plt.show()

facility_df = pd.read_parquet(
    FACILITY_STATIC_PATH,
    columns=[
        "center_lon",
        "center_lat",
        "facility_storage_count",
        "facility_agency_count",
        "facility_factory_count",
        "storage_influence",
        "agency_influence",
        "factory_influence",
    ],
)

display(
    facility_df[[
        "facility_storage_count",
        "facility_agency_count",
        "facility_factory_count",
        "storage_influence",
        "agency_influence",
        "factory_influence",
    ]].describe().T
)

plot_static_grid(
    facility_df,
    value_col="facility_storage_count",
    title="Storage Count (저유소 개수, log1p)",
    log1p=True,
    robust=False,
    cmap="viridis",
)

plot_static_grid(
    facility_df,
    value_col="facility_agency_count",
    title="Agency Count (대리점 개수, log1p)",
    log1p=True,
    robust=False,
    cmap="viridis",
)

plot_static_grid(
    facility_df,
    value_col="facility_factory_count",
    title="Factory Count (공장 개수, log1p)",
    log1p=True,
    robust=False,
    cmap="viridis",
)

plot_static_grid(
    facility_df,
    value_col="storage_influence",
    title="Storage Influence (저유소 영향력, log1p)",
    log1p=True,
    robust=True,
    cmap="YlOrRd",
)

plot_static_grid(
    facility_df,
    value_col="agency_influence",
    title="Agency Influence (대리점 영향력, log1p)",
    log1p=True,
    robust=True,
    cmap="YlOrRd",
)

plot_static_grid(
    facility_df,
    value_col="factory_influence",
    title="Factory Influence (공장 영향력, log1p)",
    log1p=True,
    robust=True,
    cmap="YlOrRd",
)

#### 섬 여부 추가

In [ ]:
# ============================================================
# 6셀. 바다/섬 플래그 추가
# - 추가 데이터 필요 없음
# - 현재 최종 land grid의 연결 컴포넌트로 island 판정
# - is_sea 는 현재 구조상 모두 0
#
# 저장 파일:
#   1) geo_flag_land_grid_500m.parquet
#   2) korea_land_grid_500m_plus_geo.parquet
#   3) facility_effect_land_grid_static_500m_plus_geo.parquet   (있으면)
#   4) grid_station_daily_panel_500m_plus_geo.parquet           (있으면)
#   5) grid_station_daily_panel_500m_plus_facility_plus_geo.parquet (있으면)
# ============================================================

from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import duckdb

# ------------------------------------------------------------
# 설정
# ------------------------------------------------------------
CELL_SIZE_M = globals().get("CELL_SIZE_M", 500)

if "DATA1_DIR" in globals():
    DATA1_DIR = Path(DATA1_DIR)
else:
    DATA1_DIR = Path(ROOT_PATH) / "그리드" / "data_1"

LAND_GRID_PATH = DATA1_DIR / f"korea_land_grid_{CELL_SIZE_M}m.parquet"

GEO_FLAG_PATH = DATA1_DIR / f"geo_flag_land_grid_{CELL_SIZE_M}m.parquet"
LAND_GRID_PLUS_GEO_PATH = DATA1_DIR / f"korea_land_grid_{CELL_SIZE_M}m_plus_geo.parquet"
FACILITY_STATIC_PATH = DATA1_DIR / f"facility_effect_land_grid_static_{CELL_SIZE_M}m.parquet"
FACILITY_STATIC_PLUS_GEO_PATH = DATA1_DIR / f"facility_effect_land_grid_static_{CELL_SIZE_M}m_plus_geo.parquet"

STATION_PANEL_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m.parquet"
STATION_PANEL_PLUS_GEO_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m_plus_geo.parquet"

STATION_FACILITY_PANEL_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m_plus_facility.parquet"
STATION_FACILITY_PANEL_PLUS_GEO_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m_plus_facility_plus_geo.parquet"

# ------------------------------------------------------------
# 함수
# ------------------------------------------------------------
def ql(s: str) -> str:
    return "'" + str(s).replace("'", "''") + "'"

def connected_components_on_grid(cell_df, cell_size_m=500):
    """
    land grid의 연결 컴포넌트 계산
    - 8방향 인접 사용
    - 가장 큰 component가 본토가 될 것
    """
    coords = list(zip(cell_df["cell_x"].astype(int), cell_df["cell_y"].astype(int)))
    coord_set = set(coords)

    visited = set()
    components = []

    neighbor_offsets = [
        (-cell_size_m, -cell_size_m),
        (-cell_size_m, 0),
        (-cell_size_m, cell_size_m),
        (0, -cell_size_m),
        (0, cell_size_m),
        (cell_size_m, -cell_size_m),
        (cell_size_m, 0),
        (cell_size_m, cell_size_m),
    ]

    for start in coords:
        if start in visited:
            continue

        queue = deque([start])
        visited.add(start)
        comp = []

        while queue:
            cur = queue.popleft()
            comp.append(cur)

            cx, cy = cur
            for dx, dy in neighbor_offsets:
                nb = (cx + dx, cy + dy)
                if nb in coord_set and nb not in visited:
                    visited.add(nb)
                    queue.append(nb)

        components.append(comp)

    return components

def build_geo_flags_from_land_grid(land_grid_df, cell_size_m=500):
    """
    현재 land grid 자체에서
    - is_sea: 모두 0
    - is_island: largest connected component만 0, 나머지 1
    """
    base = land_grid_df[["grid_id", "cell_x", "cell_y"]].drop_duplicates().copy()

    components = connected_components_on_grid(
        base[["cell_x", "cell_y"]],
        cell_size_m=cell_size_m,
    )

    # 가장 큰 component = 본토
    component_sizes = np.array([len(c) for c in components], dtype=int)
    mainland_idx = int(component_sizes.argmax())

    rows = []
    for comp_idx, comp in enumerate(components):
        is_island_val = 0 if comp_idx == mainland_idx else 1
        for x, y in comp:
            rows.append((x, y, comp_idx, is_island_val))

    geo_flag_df = pd.DataFrame(
        rows,
        columns=["cell_x", "cell_y", "land_component_id", "is_island"]
    )

    geo_flag_df["cell_x"] = geo_flag_df["cell_x"].astype("int32")
    geo_flag_df["cell_y"] = geo_flag_df["cell_y"].astype("int32")
    geo_flag_df["land_component_id"] = geo_flag_df["land_component_id"].astype("int32")
    geo_flag_df["is_island"] = geo_flag_df["is_island"].astype("int8")
    geo_flag_df["is_sea"] = np.int8(0)

    geo_flag_df = base.merge(
        geo_flag_df,
        on=["cell_x", "cell_y"],
        how="left",
    )

    return geo_flag_df, component_sizes, mainland_idx

def attach_geo_flags_to_parquet(input_path, geo_flag_path, output_path):
    input_path = Path(input_path)
    geo_flag_path = Path(geo_flag_path)
    output_path = Path(output_path)

    if not input_path.exists():
        print(f"[생략] 파일 없음: {input_path.name}")
        return

    if output_path.exists():
        output_path.unlink()

    con = duckdb.connect()

    schema_df = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet({ql(str(input_path))})"
    ).df()
    existing_cols = schema_df["column_name"].tolist()

    geo_cols = ["is_sea", "is_island", "land_component_id"]
    overlap_cols = [c for c in geo_cols if c in existing_cols]

    if overlap_cols:
        panel_sql = (
            f"SELECT * EXCLUDE ({', '.join(overlap_cols)}) "
            f"FROM read_parquet({ql(str(input_path))})"
        )
    else:
        panel_sql = f"SELECT * FROM read_parquet({ql(str(input_path))})"

    sql = f"""
    COPY (
        WITH base AS (
            {panel_sql}
        ),
        geo AS (
            SELECT
                cell_x,
                cell_y,
                is_sea,
                is_island,
                land_component_id
            FROM read_parquet({ql(str(geo_flag_path))})
        )
        SELECT
            b.*,
            CAST(COALESCE(g.is_sea, 0) AS INTEGER) AS is_sea,
            CAST(COALESCE(g.is_island, 0) AS INTEGER) AS is_island,
            CAST(g.land_component_id AS INTEGER) AS land_component_id
        FROM base b
        LEFT JOIN geo g
        USING (cell_x, cell_y)
    ) TO {ql(str(output_path))} (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    con.execute(sql)
    con.close()

    print(f"[저장] {output_path.name}")

# ------------------------------------------------------------
# 실행
# ------------------------------------------------------------
if not LAND_GRID_PATH.exists():
    raise FileNotFoundError(f"land grid 파일이 없습니다: {LAND_GRID_PATH}")

print("=" * 100)
print("[1] land grid 로드")
land_grid_df = pd.read_parquet(LAND_GRID_PATH)
print(f" - land grid 수: {len(land_grid_df):,}")

print("=" * 100)
print("[2] is_sea / is_island 생성")
geo_flag_df, component_sizes, mainland_idx = build_geo_flags_from_land_grid(
    land_grid_df=land_grid_df,
    cell_size_m=CELL_SIZE_M,
)

# 저장
geo_flag_df.to_parquet(GEO_FLAG_PATH, index=False, compression="zstd")
print(f"[저장] {GEO_FLAG_PATH.name}")

# land grid 자체에도 붙인 버전 저장
land_grid_plus_geo = land_grid_df.merge(
    geo_flag_df[["cell_x", "cell_y", "is_sea", "is_island", "land_component_id"]],
    on=["cell_x", "cell_y"],
    how="left",
)
land_grid_plus_geo.to_parquet(LAND_GRID_PLUS_GEO_PATH, index=False, compression="zstd")
print(f"[저장] {LAND_GRID_PLUS_GEO_PATH.name}")

print("=" * 100)
print("[3] component 요약")
component_summary = pd.DataFrame({
    "land_component_id": np.arange(len(component_sizes), dtype=int),
    "grid_count": component_sizes,
})
component_summary["is_mainland_component"] = (
    component_summary["land_component_id"] == mainland_idx
).astype(int)
component_summary["is_island_component"] = 1 - component_summary["is_mainland_component"]

display(component_summary.sort_values("grid_count", ascending=False).head(20))

summary_df = pd.DataFrame([{
    "total_land_grids": len(geo_flag_df),
    "mainland_grids": int((geo_flag_df["is_island"] == 0).sum()),
    "island_grids": int((geo_flag_df["is_island"] == 1).sum()),
    "sea_grids_in_current_files": int((geo_flag_df["is_sea"] == 1).sum()),
    "component_count": int(len(component_sizes)),
    "mainland_component_id": int(mainland_idx),
}])
display(summary_df)

print("=" * 100)
print("[4] 관련 파일들에 geo flag 붙이기")

attach_geo_flags_to_parquet(
    input_path=LAND_GRID_PATH,
    geo_flag_path=GEO_FLAG_PATH,
    output_path=LAND_GRID_PLUS_GEO_PATH,
)

attach_geo_flags_to_parquet(
    input_path=FACILITY_STATIC_PATH,
    geo_flag_path=GEO_FLAG_PATH,
    output_path=FACILITY_STATIC_PLUS_GEO_PATH,
)

attach_geo_flags_to_parquet(
    input_path=STATION_PANEL_PATH,
    geo_flag_path=GEO_FLAG_PATH,
    output_path=STATION_PANEL_PLUS_GEO_PATH,
)

attach_geo_flags_to_parquet(
    input_path=STATION_FACILITY_PANEL_PATH,
    geo_flag_path=GEO_FLAG_PATH,
    output_path=STATION_FACILITY_PANEL_PLUS_GEO_PATH,
)

print("=" * 100)
print("[5] 미리보기")

display(
    geo_flag_df[["grid_id", "cell_x", "cell_y", "is_sea", "is_island", "land_component_id"]]
    .head(10)
)

display(
    geo_flag_df["is_island"]
    .value_counts(dropna=False)
    .rename(index={0: "본토", 1: "섬"})
    .to_frame("grid_count")
)

print("=" * 100)
print("[설명]")
print("1) is_sea 는 현재 구조에서는 모두 0이다. (이미 육지 grid만 남겨놨기 때문)")
print("2) is_island 는 최종 land grid의 연결 컴포넌트 기준으로 생성했다.")
print("3) 가장 큰 component = 본토, 나머지 component = 섬")
print("4) 그래서 제주/울릉/독도/기타 분리된 섬들은 is_island=1 로 들어간다.")
print("5) 만약 나중에 sea grid까지 포함한 전체 grid를 다시 만들면, 그때 is_sea 가 0/1로 의미를 갖게 된다.")

#### 섬 시각화

In [ ]:
# ============================================================
# 6-1셀. 섬/바다 플래그 시각화
# - 현재 구조상 is_sea는 대부분 전부 0
# - 실질적으로는 is_island 시각화가 핵심
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CELL_SIZE_M = globals().get("CELL_SIZE_M", 500)

if "DATA1_DIR" in globals():
    DATA1_DIR = Path(DATA1_DIR)
else:
    DATA1_DIR = Path(ROOT_PATH) / "그리드" / "data_1"

CANDIDATES = [
    DATA1_DIR / f"korea_land_grid_{CELL_SIZE_M}m_plus_geo.parquet",
    DATA1_DIR / f"geo_flag_land_grid_{CELL_SIZE_M}m.parquet",
]

GEO_PATH = None
for p in CANDIDATES:
    if p.exists():
        GEO_PATH = p
        break

if GEO_PATH is None:
    raise FileNotFoundError("geo flag 파일을 찾지 못했습니다.")

geo_df = pd.read_parquet(GEO_PATH)

# center 정보가 없는 geo_flag 파일이면 land_grid_plus_geo 파일을 다시 읽음
if not set(["center_lon", "center_lat"]).issubset(geo_df.columns):
    LAND_PLUS_GEO_PATH = DATA1_DIR / f"korea_land_grid_{CELL_SIZE_M}m_plus_geo.parquet"
    if LAND_PLUS_GEO_PATH.exists():
        geo_df = pd.read_parquet(LAND_PLUS_GEO_PATH)
    else:
        raise RuntimeError("center_lon/center_lat가 있는 geo 파일이 필요합니다.")

print(f"[사용 파일] {GEO_PATH}")
display(
    geo_df[["is_sea", "is_island", "land_component_id"]]
    .describe()
    .T
)

# component 크기 요약
component_summary = (
    geo_df.groupby("land_component_id", observed=True)
    .size()
    .reset_index(name="grid_count")
    .sort_values("grid_count", ascending=False)
    .reset_index(drop=True)
)
display(component_summary.head(20))

def plot_binary_flag(df, flag_col, title, cmap="coolwarm"):
    work = df[["center_lon", "center_lat", flag_col]].dropna().copy()
    work = work.sort_values(flag_col)

    fig, ax = plt.subplots(figsize=(7.2, 9.2))
    sc = ax.scatter(
        work["center_lon"],
        work["center_lat"],
        c=work[flag_col],
        s=3.0,
        marker="s",
        linewidths=0,
        cmap=cmap,
        alpha=0.95,
        rasterized=True,
    )
    ax.set_xlim(124.0, 132.2)
    ax.set_ylim(33.0, 39.7)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title)
    ax.set_aspect(1 / np.cos(np.deg2rad(36.0)))
    plt.colorbar(sc, ax=ax, shrink=0.85)
    plt.tight_layout()
    plt.show()

def plot_top_components(df, component_summary, top_k=12):
    top_ids = component_summary.head(top_k)["land_component_id"].tolist()
    work = df[df["land_component_id"].isin(top_ids)].copy()

    # 작은 component가 위에 보이게
    id_to_rank = {cid: i for i, cid in enumerate(top_ids)}
    work["_plot_rank"] = work["land_component_id"].map(id_to_rank)
    work = work.sort_values("_plot_rank", ascending=False)

    fig, ax = plt.subplots(figsize=(7.2, 9.2))
    sc = ax.scatter(
        work["center_lon"],
        work["center_lat"],
        c=work["_plot_rank"],
        s=3.0,
        marker="s",
        linewidths=0,
        cmap="tab20",
        alpha=0.95,
        rasterized=True,
    )
    ax.set_xlim(124.0, 132.2)
    ax.set_ylim(33.0, 39.7)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Top land components (largest=mainland, others=islands)")
    ax.set_aspect(1 / np.cos(np.deg2rad(36.0)))
    plt.colorbar(sc, ax=ax, shrink=0.85)
    plt.tight_layout()
    plt.show()

plot_binary_flag(
    geo_df,
    flag_col="is_island",
    title="Island Flag Map (0=mainland, 1=island)",
    cmap="coolwarm",
)

if geo_df["is_sea"].nunique(dropna=False) > 1:
    plot_binary_flag(
        geo_df,
        flag_col="is_sea",
        title="Sea Flag Map (0=land, 1=sea)",
        cmap="Blues",
    )
else:
    print("[알림] 현재 파일은 육지 grid만 저장했기 때문에 is_sea는 전부 같은 값입니다. sea map은 생략합니다.")

plot_top_components(geo_df, component_summary, top_k=12)

#### 공시지가 추가

In [ ]:
# ============================================================
# 7셀. 최종 패널에 격자화된 공시지가 붙이기 - 정리 버전
# - 초기 land grid 사용 안 함
# - static official parquet 따로 안 만듦
# - 최종 panel parquet 하나에만 official_land_price 추가
#
# 입력:
#   1) DATA_PATH / "공지시가.csv"
#   2) DATA1_DIR / "grid_station_daily_panel_500m_plus_facility_plus_geo.parquet"
#
# 출력:
#   DATA1_DIR / "grid_station_daily_panel_500m_plus_facility_plus_geo_plus_official_price.parquet"
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import duckdb

CELL_SIZE_M = 500
DATA1_DIR = Path(ROOT_PATH) / "그리드" / "data_1"

# ------------------------------------------------------------
# 0. 경로 설정: 딱 필요한 것만
# ------------------------------------------------------------
OFFICIAL_PRICE_CSV_PATH = Path(DATA_PATH) / "공시지가.csv"

PANEL_INPUT_PATH = (
    DATA1_DIR
    / f"grid_station_daily_panel_{CELL_SIZE_M}m_plus_facility_plus_geo.parquet"
)

PANEL_OUTPUT_PATH = (
    DATA1_DIR
    / f"grid_station_daily_panel_{CELL_SIZE_M}m_plus_facility_plus_geo_plus_official_price.parquet"
)

print("[입력 파일]")
print(" - OFFICIAL_PRICE_CSV_PATH:", OFFICIAL_PRICE_CSV_PATH)
print(" - PANEL_INPUT_PATH       :", PANEL_INPUT_PATH)

print("\n[출력 파일]")
print(" - PANEL_OUTPUT_PATH      :", PANEL_OUTPUT_PATH)

if not OFFICIAL_PRICE_CSV_PATH.exists():
    raise FileNotFoundError(f"공시지가 CSV 파일 없음: {OFFICIAL_PRICE_CSV_PATH}")

if not PANEL_INPUT_PATH.exists():
    raise FileNotFoundError(f"최종 패널 parquet 파일 없음: {PANEL_INPUT_PATH}")


# ------------------------------------------------------------
# 1. 보조 함수
# ------------------------------------------------------------
def ql(s: str) -> str:
    return "'" + str(s).replace("'", "''") + "'"

def qi(s: str) -> str:
    return '"' + str(s).replace('"', '""') + '"'

def read_csv_flexible_local(path, **kwargs):
    last_error = None
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            last_error = e
    raise last_error

def parse_price_cols(cols):
    """
    p_20160921, p_20171121 같은 공시지가 snapshot 컬럼만 추출
    """
    price_cols = []
    for c in cols:
        if re.match(r"^p_\d{8}$", str(c)):
            price_cols.append(c)
    return sorted(price_cols, key=lambda x: x.replace("p_", ""))

def date_yyyymmdd_to_sql_date(s):
    s = str(s)
    return f"{s[:4]}-{s[4:6]}-{s[6:8]}"

def build_asof_price_sql(price_cols_renamed, fill_before_first=False):
    """
    panel.date 기준으로 그 날짜 이하의 최신 공시지가를 선택하는 SQL CASE 생성.

    예:
      date = 2024-01-15
      사용 컬럼 = official_land_price__20231221

    각 snapshot 컬럼이 NULL이면 그 이전 snapshot 중 최신 non-null 값을 사용.
    """
    date_keys = [
        c.replace("official_land_price__", "")
        for c in price_cols_renamed
    ]

    price_when_parts = []
    source_date_when_parts = []

    # 최신 snapshot부터 CASE WHEN 생성
    for i in range(len(date_keys) - 1, -1, -1):
        dt_key = date_keys[i]
        sql_dt = date_yyyymmdd_to_sql_date(dt_key)

        # 현재 날짜 이하 snapshot들 중 최신 non-null
        candidate_cols = price_cols_renamed[: i + 1][::-1]

        coalesce_expr = "COALESCE(" + ", ".join(
            [f"o.{qi(c)}" for c in candidate_cols]
        ) + ")"

        source_date_expr_parts = []
        for c in candidate_cols:
            c_dt = c.replace("official_land_price__", "")
            c_sql_dt = date_yyyymmdd_to_sql_date(c_dt)
            source_date_expr_parts.append(
                f"WHEN o.{qi(c)} IS NOT NULL THEN DATE {ql(c_sql_dt)}"
            )

        source_date_expr = (
            "CASE "
            + " ".join(source_date_expr_parts)
            + " ELSE NULL END"
        )

        price_when_parts.append(
            f"WHEN CAST(p.date AS DATE) >= DATE {ql(sql_dt)} THEN {coalesce_expr}"
        )
        source_date_when_parts.append(
            f"WHEN CAST(p.date AS DATE) >= DATE {ql(sql_dt)} THEN {source_date_expr}"
        )

    if fill_before_first:
        first_col = price_cols_renamed[0]
        first_dt = first_col.replace("official_land_price__", "")
        first_sql_dt = date_yyyymmdd_to_sql_date(first_dt)

        price_else = f"o.{qi(first_col)}"
        source_date_else = (
            f"CASE WHEN o.{qi(first_col)} IS NOT NULL "
            f"THEN DATE {ql(first_sql_dt)} ELSE NULL END"
        )
    else:
        price_else = "NULL"
        source_date_else = "NULL"

    price_case_sql = (
        "CASE "
        + " ".join(price_when_parts)
        + f" ELSE {price_else} END"
    )

    source_date_case_sql = (
        "CASE "
        + " ".join(source_date_when_parts)
        + f" ELSE {source_date_else} END"
    )

    return price_case_sql, source_date_case_sql


# ------------------------------------------------------------
# 2. 공시지가 CSV 로드 및 정리
# ------------------------------------------------------------
print("=" * 100)
print("[1] 공시지가 CSV 로드")

official_raw = read_csv_flexible_local(OFFICIAL_PRICE_CSV_PATH)

required_cols = {"cell_x", "cell_y"}
missing = required_cols - set(official_raw.columns)
if missing:
    raise ValueError(f"공시지가 CSV 필수 컬럼 누락: {missing}")

price_cols = parse_price_cols(official_raw.columns)
if not price_cols:
    raise ValueError("p_YYYYMMDD 형태의 공시지가 컬럼을 찾지 못했습니다.")

# cell_x, cell_y 중복이면 panel join 시 행이 늘어나므로 반드시 막아야 함
dup_cell = int(official_raw[["cell_x", "cell_y"]].duplicated().sum())
if dup_cell > 0:
    raise ValueError(f"공시지가 cell_x/cell_y 중복이 있습니다: {dup_cell:,}건")

# grid_id는 join에 필수는 아니지만, 있으면 검증만 수행
if "grid_id" in official_raw.columns:
    calc_grid_id = (
        f"G{CELL_SIZE_M}_"
        + (official_raw["cell_x"].astype("int64") // CELL_SIZE_M).astype(str)
        + "_"
        + (official_raw["cell_y"].astype("int64") // CELL_SIZE_M).astype(str)
    )

    mismatch = int((official_raw["grid_id"].astype(str) != calc_grid_id).sum())
    if mismatch > 0:
        print(f"[주의] grid_id와 cell_x/cell_y가 안 맞는 행: {mismatch:,}건")
        print("       join은 cell_x/cell_y 기준으로 진행합니다.")

rename_price_cols = {
    c: f"official_land_price__{c.replace('p_', '')}"
    for c in price_cols
}

official = official_raw[["cell_x", "cell_y"] + price_cols].rename(
    columns=rename_price_cols
).copy()

official["cell_x"] = official["cell_x"].astype("int32")
official["cell_y"] = official["cell_y"].astype("int32")

official_price_cols = list(rename_price_cols.values())

for c in official_price_cols:
    # 콤마가 들어간 숫자도 대응
    s = official[c].astype(str).str.replace(",", "", regex=False)
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    official[c] = pd.to_numeric(s, errors="coerce").astype("float32")

print(f" - 공시지가 grid 수: {len(official):,}")
print(f" - 공시지가 snapshot 컬럼 수: {len(official_price_cols)}")
print(" - snapshot 컬럼:")
for c in official_price_cols:
    print("   ", c)

display(
    pd.DataFrame({
        "price_col": official_price_cols,
        "not_null_count": [int(official[c].notna().sum()) for c in official_price_cols],
        "null_count": [int(official[c].isna().sum()) for c in official_price_cols],
    })
)


# ------------------------------------------------------------
# 3. 최종 패널에 날짜 기준 최신 공시지가 붙이기
# ------------------------------------------------------------
print("=" * 100)
print("[2] 최종 패널에 공시지가 붙이기")

# False면 첫 공시지가 이전 날짜는 NULL
# True면 첫 공시지가를 2008~첫 snapshot 이전에도 강제로 넣음
FILL_BEFORE_FIRST_SNAPSHOT = False

price_case_sql, source_date_case_sql = build_asof_price_sql(
    official_price_cols,
    fill_before_first=FILL_BEFORE_FIRST_SNAPSHOT,
)

con = duckdb.connect()
con.register("official_price_grid", official)

# panel 필수 컬럼 확인
panel_schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet({ql(str(PANEL_INPUT_PATH))})"
).df()

panel_cols = panel_schema["column_name"].tolist()

required_panel_cols = {"date", "cell_x", "cell_y"}
missing_panel_cols = required_panel_cols - set(panel_cols)
if missing_panel_cols:
    con.close()
    raise ValueError(f"패널 필수 컬럼 누락: {missing_panel_cols}")

# 이미 붙인 컬럼이 있으면 제거하고 새로 붙임
new_cols = [
    "official_land_price",
    "official_price_source_date",
]

overlap_cols = [c for c in new_cols if c in panel_cols]

if overlap_cols:
    panel_sql = (
        f"SELECT * EXCLUDE ({', '.join(qi(c) for c in overlap_cols)}) "
        f"FROM read_parquet({ql(str(PANEL_INPUT_PATH))})"
    )
else:
    panel_sql = f"SELECT * FROM read_parquet({ql(str(PANEL_INPUT_PATH))})"

if PANEL_OUTPUT_PATH.exists():
    PANEL_OUTPUT_PATH.unlink()

sql = f"""
COPY (
    WITH panel AS (
        {panel_sql}
    ),
    official AS (
        SELECT *
        FROM official_price_grid
    )
    SELECT
        p.*,
        CAST({price_case_sql} AS FLOAT) AS official_land_price,
        CAST({source_date_case_sql} AS DATE) AS official_price_source_date
    FROM panel p
    LEFT JOIN official o
    USING (cell_x, cell_y)
) TO {ql(str(PANEL_OUTPUT_PATH))}
(FORMAT PARQUET, COMPRESSION ZSTD)
"""

con.execute(sql)

print(f"[저장] {PANEL_OUTPUT_PATH}")


# ------------------------------------------------------------
# 4. 검증 요약
# ------------------------------------------------------------
print("=" * 100)
print("[3] 검증 요약")

coverage_df = con.execute(f"""
    WITH panel_grid AS (
        SELECT DISTINCT cell_x, cell_y
        FROM read_parquet({ql(str(PANEL_INPUT_PATH))})
    ),
    official_grid AS (
        SELECT DISTINCT cell_x, cell_y
        FROM official_price_grid
    )
    SELECT
        COUNT(*) AS panel_unique_grid_count,
        SUM(CASE WHEN o.cell_x IS NOT NULL THEN 1 ELSE 0 END) AS matched_official_grid_count,
        SUM(CASE WHEN o.cell_x IS NULL THEN 1 ELSE 0 END) AS unmatched_official_grid_count
    FROM panel_grid p
    LEFT JOIN official_grid o
    USING (cell_x, cell_y)
""").df()

summary_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(CAST(date AS DATE)) AS min_date,
        MAX(CAST(date AS DATE)) AS max_date,
        COUNT(DISTINCT grid_id) AS unique_grid_count,
        SUM(CASE WHEN official_land_price IS NULL THEN 1 ELSE 0 END) AS official_price_null_rows,
        AVG(official_land_price) AS official_price_avg,
        MIN(official_land_price) AS official_price_min,
        MAX(official_land_price) AS official_price_max
    FROM read_parquet({ql(str(PANEL_OUTPUT_PATH))})
""").df()

source_date_summary_df = con.execute(f"""
    SELECT
        official_price_source_date,
        COUNT(*) AS row_count,
        AVG(official_land_price) AS avg_official_land_price
    FROM read_parquet({ql(str(PANEL_OUTPUT_PATH))})
    GROUP BY 1
    ORDER BY 1
""").df()

preview_df = con.execute(f"""
    SELECT
        date,
        grid_id,
        cell_x,
        cell_y,
        official_land_price,
        official_price_source_date
    FROM read_parquet({ql(str(PANEL_OUTPUT_PATH))})
    LIMIT 10
""").df()

con.close()

print("\n[grid coverage]")
display(coverage_df)

print("\n[panel summary]")
display(summary_df)

print("\n[source date summary]")
display(source_date_summary_df)

print("\n[preview]")
display(preview_df)

In [ ]:
# ============================================================
# 7-1셀. 공시지가 데이터 시각화
# - 특정 날짜 1개
# - 2개 그림:
#   1) official_land_price
#   2) official_land_price log1p
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

CELL_SIZE_M = 500
DATA1_DIR = Path(ROOT_PATH) / "그리드" / "data_1"

def ql(s: str) -> str:
    return "'" + str(s).replace("'", "''") + "'"

PANEL_PATH = DATA1_DIR / f"grid_station_daily_panel_{CELL_SIZE_M}m_plus_facility_plus_geo_plus_official_price.parquet"

# 여기만 바꿔서 보면 됨
CHECK_DATE = "2024-08-02"

def auto_point_size(n):
    if n >= 250000:
        return 1.0
    elif n >= 120000:
        return 2.0
    elif n >= 50000:
        return 3.5
    else:
        return 6.0

def plot_grid_map(df, value_col, title, log1p=False, robust=False, cmap="YlOrRd"):
    work = df[["center_lon", "center_lat", value_col]].dropna().copy()
    if len(work) == 0:
        print(f"[plot skip] {value_col}: 데이터 없음")
        return

    values = work[value_col].astype(float).to_numpy()

    if log1p:
        values = np.log1p(np.maximum(values, 0))

    vmin, vmax = None, None
    finite_vals = values[np.isfinite(values)]

    if robust and len(finite_vals) >= 20:
        vmin, vmax = np.nanpercentile(finite_vals, [1, 99])
        if np.isclose(vmin, vmax):
            vmin, vmax = None, None

    work = work.assign(_plot_value=values).sort_values("_plot_value")

    fig, ax = plt.subplots(figsize=(7.2, 9.2))
    sc = ax.scatter(
        work["center_lon"],
        work["center_lat"],
        c=work["_plot_value"],
        s=auto_point_size(len(work)),
        marker="s",
        linewidths=0,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        alpha=0.95,
        rasterized=True,
    )

    ax.set_xlim(124.0, 132.2)
    ax.set_ylim(33.0, 39.7)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title)
    ax.set_aspect(1 / np.cos(np.deg2rad(36.0)))
    plt.colorbar(sc, ax=ax, shrink=0.85)
    plt.tight_layout()
    plt.show()

if not PANEL_PATH.exists():
    raise FileNotFoundError(f"파일 없음: {PANEL_PATH}")

con = duckdb.connect()

plot_df = con.execute(f"""
    SELECT
        CAST(date AS DATE) AS date,
        grid_id,
        cell_x,
        cell_y,
        center_lon,
        center_lat,
        station_count_total,
        gasoline_price_mean,
        official_land_price,
        official_price_source_date
    FROM read_parquet({ql(str(PANEL_PATH))})
    WHERE CAST(date AS DATE) = CAST({ql(CHECK_DATE)} AS DATE)
""").df()

con.close()

if len(plot_df) == 0:
    raise RuntimeError(f"{CHECK_DATE} 날짜의 데이터가 없습니다.")

print(f"[사용 파일] {PANEL_PATH}")
print(f"[선택 날짜] {CHECK_DATE}")
print(f"[row 수] {len(plot_df):,}")

display(
    plot_df[[
        "station_count_total",
        "gasoline_price_mean",
        "official_land_price",
    ]]
    .describe()
    .T
)

display(
    plot_df["official_price_source_date"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("official_price_source_date")
    .reset_index(name="row_count")
)

plot_grid_map(
    plot_df,
    value_col="official_land_price",
    title=f"{CHECK_DATE} | Official Land Price",
    log1p=False,
    robust=True,
    cmap="viridis",
)

plot_grid_map(
    plot_df,
    value_col="official_land_price",
    title=f"{CHECK_DATE} | Official Land Price (log1p)",
    log1p=True,
    robust=True,
    cmap="YlOrRd",
)